# 15 — V2 Survival Analysis · Advanced Models + Calibration

Tune and calibrate **advanced leakage-safe time-to-next-service survival models** on FROZEN
RideBase v1.3. Candidates: **CoxNet** (elastic-net Cox) · **XGBoost `survival:cox`** · **XGBoost
`survival:aft`**, with a **Random Survival Forest** ranking diagnostic and a
**GradientBoostingSurvival** attempt under a wall-clock budget guard.

Selection is on a **VALIDATION composite** (40% IBS · 25% Brier@90 · 20% (1−IPCW-C) · 15%
calibration@90). Probabilities are calibrated with **per-horizon IPCW-weighted isotonic
regression fit on VALIDATION only**. **TEST is opened once**, after the full configuration —
model family, feature set, hyper-parameters, calibration maps, ensemble weights, horizons,
risk thresholds — is frozen. The **nb14 Cox baseline stays champion unless an advanced model
beats it meaningfully** (≥5% IBS or Brier@90, or calibration MODERATE→GOOD). Dataset,
generator, split, target and censoring are **not** changed; no re-tuning against TEST.

In [1]:
"""15_v2_survival_advanced — tune + calibrate advanced leakage-safe time-to-next-service
survival models on FROZEN RideBase v1.3, optimising 30/60/90/120-day probability quality,
calibration, temporal generalisation and motorcycle-grouped evaluation.

Candidates: CoxNet (regularised Cox) · XGBoost survival:cox · XGBoost survival:aft
(+ RSF ranking diagnostic, GradientBoostingSurvival under a time-budget guard).
Selection priority: calibration > IBS > horizon Brier > IPCW C-index > AUC > temporal stability.
Dataset / generator / split / target / censoring are NOT changed. TEST opened once, after the
full configuration (model family, feature set, hyper-params, calibration, ensemble weights,
horizons, risk thresholds) is frozen on VALIDATION. Advanced model kept only if it beats the
Cox baseline meaningfully; otherwise the Cox baseline stays champion."""
from pathlib import Path
import hashlib, json, os, time, warnings, threading

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sksurv.util import Surv
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import (concordance_index_censored, concordance_index_ipcw,
                            brier_score, integrated_brier_score, cumulative_dynamic_auc)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 200); pd.set_option("display.width", 220)

SEED = 42
np.random.seed(SEED)
FAST_MODE = os.environ.get("RB_FAST") == "1"
DATASET_VERSION = "1.3.0"
HORIZONS = [30, 60, 90, 120]
DIAG_HORIZONS = [180]
ALL_H = HORIZONS + DIAG_HORIZONS
N_TRIALS = 15 if FAST_MODE else 120          # FULL: target 100–150, min 75 successful
N_BOOT = 40 if FAST_MODE else 500            # FULL: motorcycle-grouped bootstrap replicates
N_FOLDS = 2 if FAST_MODE else 3
XGB_ROUNDS = 400 if FAST_MODE else 700       # n_estimators search ceiling
GBS_BUDGET_S = 90 if FAST_MODE else 210
XGB_STUDY_TIMEOUT = None if FAST_MODE else 2000   # per-study wall-clock cap (s); n_trials OR timeout, whichever first
RSF_BENCH_TIMEOUT = 40
RUN_MODE = "FAST" if FAST_MODE else "FULL"
TGRID = np.arange(1, 1001)          # day grid for median-service-day search
# composite validation score (lower = better) — natural metric scales, documented in §24
W_IBS, W_B90, W_IPCWC, W_CAL = 0.40, 0.25, 0.20, 0.15

def find_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "notebooks").is_dir() and (c / "models").is_dir() and (c / "outputs").is_dir():
            return c
    raise FileNotFoundError("ridebase-ml root not found")

ROOT = find_root()
OUTPUTS, MODELS, REPORTS = ROOT / "outputs", ROOT / "models", ROOT / "reports"
TABLES, FIGS = REPORTS / "tables", REPORTS / "figures" / "v2_survival_advanced"
for d in (MODELS, TABLES, FIGS, OUTPUTS):
    d.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS / name, dpi=140, bbox_inches="tight"); plt.close()

QA = []
def qa(check, value, expected, ok, notes=""):
    QA.append({"check": check, "value": str(value)[:120], "expected": str(expected),
               "status": "PASS" if ok else "FAIL", "notes": notes})
    print(f"  [{'PASS' if ok else 'FAIL'}] {check}: {str(value)[:90]} (exp {expected}) {notes}")

plt.style.use("seaborn-v0_8-whitegrid")
print(f"SETUP OK | FAST_MODE={FAST_MODE} | trials/model={N_TRIALS} | folds={N_FOLDS} | boot={N_BOOT}")

/Users/nihatkutukoglu/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SETUP OK | FAST_MODE=False | trials/model=120 | folds=3 | boot=500


## 1 · Data guard
Load `outputs/v2_survival_modeling_table.parquet` (nb13). Assert dataset version 1.3.x,
split `{TRAIN 32203, VALIDATION 4845, TEST 4470}`, `duration_days > 0`,
`event_observed ∈ {0,1}`, no missing target; record the input hash. Merge `snapshot_at` and
the audit-only next-event type from the target-audit table.

In [2]:
MT_PATH = OUTPUTS / "v2_survival_modeling_table.parquet"
TA_PATH = OUTPUTS / "v2_survival_target_audit.parquet"
mt = pd.read_parquet(MT_PATH)
ta = pd.read_parquet(TA_PATH)
INPUT_HASH = hashlib.sha256(MT_PATH.read_bytes()).hexdigest()[:16]

EXP_SPLIT = {"TRAIN": 32203, "VALIDATION": 4845, "TEST": 4470}
got = mt.split.value_counts().to_dict()
qa("dataset_version", DATASET_VERSION, "1.3 / 1.3.0", DATASET_VERSION.startswith("1.3"))
qa("split_unchanged", got, EXP_SPLIT, got == EXP_SPLIT)
qa("survival_rows", len(mt), 41518, len(mt) == 41518)
qa("duration_positive", int((mt.duration_days <= 0).sum()), 0, (mt.duration_days <= 0).sum() == 0)
qa("event_binary", sorted(mt.event_observed.unique().tolist()), "[0, 1]",
   sorted(mt.event_observed.unique().tolist()) == [0, 1])
qa("no_missing_target", int(mt[["duration_days", "event_observed"]].isna().sum().sum()), 0,
   mt[["duration_days", "event_observed"]].isna().sum().sum() == 0)
qa("input_hash", INPUT_HASH, "stable", True, "sha256[:16] of modeling table")

mt = mt.merge(ta[["snapshot_id", "snapshot_at", "next_event_type_audit"]], on="snapshot_id", how="left")
mt["snapshot_at"] = pd.to_datetime(mt["snapshot_at"])
N_EVENTS = int(mt.event_observed.sum()); N_CENSORED = int((mt.event_observed == 0).sum())
print(f"rows {len(mt)} | events {N_EVENTS} | censored {N_CENSORED} | motorcycles {mt.motorcycle_id.nunique()}")
print("censoring rate by split:", (1 - mt.groupby('split').event_observed.mean()).round(3).to_dict())

  [PASS] dataset_version: 1.3.0 (exp 1.3 / 1.3.0) 
  [PASS] split_unchanged: {'TRAIN': 32203, 'VALIDATION': 4845, 'TEST': 4470} (exp {'TRAIN': 32203, 'VALIDATION': 4845, 'TEST': 4470}) 
  [PASS] survival_rows: 41518 (exp 41518) 
  [PASS] duration_positive: 0 (exp 0) 
  [PASS] event_binary: [0, 1] (exp [0, 1]) 
  [PASS] no_missing_target: 0 (exp 0) 
  [PASS] input_hash: dfc4ee1b266850dd (exp stable) sha256[:16] of modeling table
rows 41518 | events 28153 | censored 13365 | motorcycles 8442
censoring rate by split: {'TEST': 0.713, 'TRAIN': 0.21, 'VALIDATION': 0.705}


## 2 · Target + feature contract
Survival target = `(duration_days, event_observed)`; right-censored rows used **natively**.
Features = leakage-safe snapshot columns; `next_service*`, `next_event*`, `censor*`,
`future_*`, `*_audit`, ids, `service_sequence` and the target are excluded by name and
substring. `motorcycle_id` is used **only for grouping**; episode rank per motorcycle drives
the FIRST/LAST-episode diagnostics.

In [3]:
FORBIDDEN = {"duration_days", "event_observed", "snapshot_id", "motorcycle_id", "customer_id",
             "service_id", "is_right_censored", "censoring_date", "censoring_boundary", "snapshot_at",
             "next_service_at", "next_event_type", "next_event_type_audit", "target_source",
             "label_contract", "service_sequence"}
FORBID_SUBSTR = ("next_service", "next_event", "censor", "future_", "duration_days", "_audit")
FEATURES = [c for c in mt.columns if c not in FORBIDDEN and not any(s in c for s in FORBID_SUBSTR)]
CAT_FEATURES = [c for c in FEATURES if str(mt[c].dtype) in ("object", "category", "bool")]
NUM_FEATURES = [c for c in FEATURES if c not in CAT_FEATURES]
assert not (set(FEATURES) & FORBIDDEN)
qa("no_target_leakage_in_features", len(set(FEATURES) & FORBIDDEN), 0, len(set(FEATURES) & FORBIDDEN) == 0,
   f"{len(FEATURES)} features ({len(NUM_FEATURES)} num / {len(CAT_FEATURES)} cat)")
qa("no_future_information", "next_*/censor*/future_* excluded by substring guard", "yes", True)

masks = {s: (mt.split == s).to_numpy() for s in ("TRAIN", "VALIDATION", "TEST")}
def y_of(m):
    return Surv.from_arrays(event=mt.loc[m, "event_observed"].to_numpy().astype(bool),
                            time=mt.loc[m, "duration_days"].to_numpy().astype(float))
Y = {s: y_of(masks[s]) for s in masks}
GROUPS = {s: mt.loc[masks[s], "motorcycle_id"].to_numpy() for s in masks}
DUR = {s: mt.loc[masks[s], "duration_days"].to_numpy().astype(float) for s in masks}
EV = {s: mt.loc[masks[s], "event_observed"].to_numpy().astype(int) for s in masks}
SNAP = {s: mt.loc[masks[s], "snapshot_at"].to_numpy() for s in masks}
ep_rank = (mt.assign(_o=np.arange(len(mt)))
             .sort_values(["motorcycle_id", "service_sequence"])
             .groupby("motorcycle_id").cumcount())
mt["_ep_rank"] = ep_rank.reindex(mt.index).values
mt["_is_first_ep"] = mt._ep_rank == 0
mt["_is_last_ep"] = mt.groupby("motorcycle_id")._ep_rank.transform("max") == mt._ep_rank
print(f"features {len(FEATURES)} | TEST first-ep {int(mt.loc[masks['TEST'],'_is_first_ep'].sum())}"
      f" / last-ep {int(mt.loc[masks['TEST'],'_is_last_ep'].sum())}")

  [PASS] no_target_leakage_in_features: 0 (exp 0) 145 features (119 num / 26 cat)
  [PASS] no_future_information: next_*/censor*/future_* excluded by substring guard (exp yes) 


features 145 | TEST first-ep 857 / last-ep 3188


## 3 · Feature sets + train-only preprocessing
`SET_A_COMPACT` · `SET_B_HISTORY_USAGE` · `SET_C_FULL_LEAKAGE_SAFE` · `SET_D_TARGET_SPECIFIC_SURVIVAL`
(SET_C with `|corr|>0.95` redundancy pruning + near-zero-variance drop, **computed on TRAIN
only**). A default ridge-Cox picks the set by VALIDATION IPCW-C with a parsimony tie-break
(smaller set wins within 0.01). Preprocessing (`ColumnTransformer`, fit on TRAIN only):
numeric → median impute + indicator + standardize; categorical → `UNKNOWN` → one-hot
(`min_frequency=30`). No TEST, no high-cardinality identifiers.

In [4]:
COMPACT = [c for c in ["snapshot_year", "snapshot_month", "motorcycle_age_years", "engine_displacement_cc",
    "brand", "category", "usage_type", "riding_intensity", "policy_interval_km", "policy_interval_days",
    "previous_service_count", "days_since_previous_service", "km_since_previous_service",
    "historical_interval_days_median", "historical_interval_km_median", "recent_90d_km",
    "snapshot_odometer_km", "annual_km_baseline"] if c in FEATURES]
HIST_USAGE = [c for c in FEATURES if any(k in c for k in
    ("recent_", "historical_interval", "previous_interval", "avg_service_interval", "rolling3",
     "maintenance", "policy_", "services_last", "days_since_", "km_since_", "history"))
    or c in ("annual_km_baseline", "riding_intensity", "usage_type", "load_severity_factor",
             "motorcycle_age_years", "snapshot_odometer_km")]

def make_pre(cols):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                          ("sc", StandardScaler())]), [c for c in cols if c in NUM_FEATURES]),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
                          ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30,
                                              sparse_output=False))]), [c for c in cols if c in CAT_FEATURES]),
    ], remainder="drop", verbose_feature_names_out=True)

def enc_subset(cols):
    keep = [c for c in cols if c in FEATURES]
    sub = make_pre(keep)
    sub.fit(mt.loc[masks["TRAIN"], keep].astype({c: str for c in keep if c in CAT_FEATURES}))
    X = {s: sub.transform(mt.loc[masks[s], keep].astype({c: str for c in keep if c in CAT_FEATURES})).astype(np.float64)
         for s in masks}
    return X, sub

# SET_D — correlation-pruned SET_C (TRAIN only): drop one of every |corr|>0.95 pair + near-zero variance
Xc_tr = mt.loc[masks["TRAIN"], [c for c in FEATURES if c in NUM_FEATURES]].apply(pd.to_numeric, errors="coerce")
cm = Xc_tr.corr().abs()
drop = set()
for i, a in enumerate(cm.columns):
    if a in drop: continue
    for b in cm.columns[i + 1:]:
        if b not in drop and cm.loc[a, b] > 0.95:
            drop.add(b)
lowvar = {c for c in cm.columns if Xc_tr[c].std(skipna=True) < 1e-9}
SET_D = [c for c in FEATURES if c not in drop and c not in lowvar]

FEATURE_SETS = {"SET_A_COMPACT": COMPACT, "SET_B_HISTORY_USAGE": sorted(set(HIST_USAGE)),
                "SET_C_FULL_LEAKAGE_SAFE": FEATURES, "SET_D_TARGET_SPECIFIC_SURVIVAL": SET_D}
fs_rows = []
for name, cols in FEATURE_SETS.items():
    Xs, _ = enc_subset(cols)
    try:
        cmod = CoxPHSurvivalAnalysis(alpha=1.0).fit(Xs["TRAIN"], Y["TRAIN"])
        risk = cmod.predict(Xs["VALIDATION"])
        ic = float(concordance_index_ipcw(Y["TRAIN"], Y["VALIDATION"], risk, tau=max(HORIZONS))[0])
    except Exception as e:
        ic = np.nan; print(f"  {name}: ridge-Cox failed ({e})")
    fs_rows.append({"feature_set": name, "n_cols": len([c for c in cols if c in FEATURES]),
                    "encoded_dims": Xs["TRAIN"].shape[1], "ridge_cox_val_ipcw_c": round(float(ic), 4)})
FS_TABLE = pd.DataFrame(fs_rows)
print(FS_TABLE.to_string(index=False))
_best = FS_TABLE.ridge_cox_val_ipcw_c.max()
BEST_FS = FS_TABLE[FS_TABLE.ridge_cox_val_ipcw_c >= _best - 0.01].sort_values("n_cols").iloc[0]["feature_set"]
Xb, PRE = enc_subset(FEATURE_SETS[BEST_FS])
ENC_NAMES = [n.replace("num__", "").replace("cat__", "") for n in PRE.get_feature_names_out()]
qa("feature_selection_train_val_only", "ridge-Cox IPCW-C on VALIDATION + parsimony tie-break", "no TEST", True)
qa("no_high_cardinality_identifier", "ids excluded; OHE min_frequency=30", "yes", True)
print(f"selected feature set (VALIDATION, parsimony tie-break): {BEST_FS} | encoded dims {Xb['TRAIN'].shape[1]}")

                   feature_set  n_cols  encoded_dims  ridge_cox_val_ipcw_c
                 SET_A_COMPACT      17            51                0.9092
           SET_B_HISTORY_USAGE      52            95                0.9092
       SET_C_FULL_LEAKAGE_SAFE     145           280                0.9189
SET_D_TARGET_SPECIFIC_SURVIVAL     117           243                0.9202


  [PASS] feature_selection_train_val_only: ridge-Cox IPCW-C on VALIDATION + parsimony tie-break (exp no TEST) 
  [PASS] no_high_cardinality_identifier: ids excluded; OHE min_frequency=30 (exp yes) 
selected feature set (VALIDATION, parsimony tie-break): SET_D_TARGET_SPECIFIC_SURVIVAL | encoded dims 243


## 4 · Kaplan-Meier reference + IPCW weights
Population `S(t)` on TRAIN (naive non-personalised baseline). Censoring-time Kaplan-Meier
`Ĝ(t)` on TRAIN supplies the inverse-probability-of-censoring weights used by the
censoring-aware calibration.

In [5]:
km_t, km_s = kaplan_meier_estimator(Y["TRAIN"]["event"], Y["TRAIN"]["time"])
def km_S(t):
    idx = np.searchsorted(km_t, t, side="right") - 1
    return float(km_s[idx]) if idx >= 0 else 1.0
KM_RISK = {h: 1 - km_S(h) for h in ALL_H}
KM_MEDIAN = float(km_t[np.argmax(km_s <= 0.5)]) if (km_s <= 0.5).any() else np.nan
KM_S_ALLH = np.array([km_S(t) for t in ALL_H])
# censoring-distribution KM (Kaplan-Meier of the censoring times) on TRAIN — for IPCW calibration weights
g_t, g_s = kaplan_meier_estimator(~Y["TRAIN"]["event"].astype(bool), Y["TRAIN"]["time"])
def G_hat(t):
    idx = np.searchsorted(g_t, t, side="right") - 1
    return max(float(g_s[idx]) if idx >= 0 else 1.0, 1e-3)
print("KM reference 1-S:", {h: round(v, 3) for h, v in KM_RISK.items()},
      "| median", f"{KM_MEDIAN:.0f}d" if np.isfinite(KM_MEDIAN) else "not reached")

KM reference 1-S: {30: 0.049, 60: 0.247, 90: 0.394, 120: 0.506, 180: 0.66} | median 118d


## 5 · Censoring-aware metrics
Harrell C-index + **Uno / IPCW C-index** (censoring distribution from **TRAIN**). Brier per
horizon + **Integrated Brier Score**. **Time-dependent AUC**. **KM-adjusted calibration
error** per horizon (10 predicted-risk bins; observed risk = `1 − KM(h)` within bin so
censored-before-`h` rows are handled correctly). The **validation composite** (lower = better)
blends IBS / Brier@90 / (1−IPCW-C) / calibration@90 on their natural scales.

In [6]:
def _auc_times(y_tr, y_ev, times):
    fmax = y_ev["time"][y_ev["event"]].max() if y_ev["event"].any() else y_ev["time"].max()
    return [t for t in times if t < fmax and (y_ev["time"] > t).sum() >= 20]

def cal_error(risk_h, dur, ev, h, nbins=10):
    """IPCW/KM-adjusted calibration error at one horizon: 10 predicted-risk bins,
    observed = 1 - KM(h) within bin (censored-before-h handled by the KM estimate)."""
    try:
        bins = pd.qcut(risk_h, nbins, duplicates="drop")
    except ValueError:
        bins = pd.cut(risk_h, nbins)
    df = pd.DataFrame({"p": risk_h, "d": dur, "e": ev, "b": bins})
    gaps, rows = [], []
    for b, g in df.groupby("b", observed=True):
        if len(g) < 5:
            continue
        obs = float(1 - KaplanMeierFitter().fit(g.d, g.e).predict(h))
        gaps.append(abs(g.p.mean() - obs))
        rows.append({"horizon": h, "bin": str(b), "n": int(len(g)),
                     "mean_predicted": round(float(g.p.mean()), 4), "observed_km": round(obs, 4),
                     "absolute_gap": round(abs(g.p.mean() - obs), 4)})
    return (float(np.mean(gaps)) if gaps else np.nan), rows

def evaluate(name, S_allh, y_tr, y_ev, dur, ev):
    """S_allh: (n, len(ALL_H)) survival probabilities on the FULL split. Risk score = 1 - S(120)."""
    S = np.clip(S_allh, 0.0, 1.0)
    risk = 1 - S[:, ALL_H.index(max(HORIZONS))]
    r = {"model": name, "n": int(len(y_ev)), "events": int(y_ev["event"].sum())}
    try:
        r["c_index"] = float(concordance_index_censored(y_ev["event"], y_ev["time"], risk)[0])
    except Exception:
        r["c_index"] = np.nan
    try:
        r["ipcw_c_index"] = float(concordance_index_ipcw(y_tr, y_ev, risk, tau=max(HORIZONS))[0])
    except Exception:
        r["ipcw_c_index"] = np.nan
    for h in ALL_H:
        r[f"brier_{h}"] = np.nan; r[f"auc_{h}"] = np.nan; r[f"cal_error_{h}"] = np.nan
    tta = _auc_times(y_tr, y_ev, ALL_H)
    try:
        auc, _ = cumulative_dynamic_auc(y_tr, y_ev, risk, tta)
        for t, a in zip(tta, np.atleast_1d(auc)):
            r[f"auc_{t}"] = float(a)
    except Exception:
        pass
    ttb = [t for t in ALL_H if t < (y_ev["time"][y_ev["event"]].max() if y_ev["event"].any() else 0)
           and (y_ev["time"] > t).sum() >= 20]
    if ttb:
        cols = [ALL_H.index(t) for t in ttb]
        try:
            _, bs = brier_score(y_tr, y_ev, S[:, cols], ttb)
            for t, b in zip(ttb, bs):
                r[f"brier_{t}"] = float(b)
        except Exception:
            pass
        try:
            r["ibs"] = float(integrated_brier_score(y_tr, y_ev, S[:, cols], ttb)) if len(ttb) > 1 else np.nan
        except Exception:
            r["ibs"] = np.nan
    else:
        r["ibs"] = np.nan
    for h in HORIZONS:
        r[f"cal_error_{h}"], _ = cal_error(1 - S[:, ALL_H.index(h)], dur, ev, h)
    return r

def composite(row):
    def sg(k, d):
        v = row.get(k, np.nan); return d if v is None or not np.isfinite(v) else v
    return (W_IBS * sg("ibs", 0.25) + W_B90 * sg("brier_90", 0.20)
            + W_IPCWC * (1 - sg("ipcw_c_index", 0.5)) + W_CAL * sg("cal_error_90", 0.15))

def check_monotone(risk_mat, tag):
    hc = [ALL_H.index(h) for h in HORIZONS]
    bad_mono = int((np.diff(risk_mat[:, hc], axis=1) < -1e-9).any(axis=1).sum())
    bad_bound = int(((risk_mat < -1e-9) | (risk_mat > 1 + 1e-9)).any(axis=1).sum())
    qa(f"prob_monotonic_{tag}", bad_mono, 0, bad_mono == 0)
    qa(f"prob_bounds_{tag}", bad_bound, 0, bad_bound == 0)

## 6 · Survival-function helpers
**CoxNet** → `predict_survival_function`. **XGB `survival:cox`** → Breslow cumulative baseline
hazard `H₀(t)` from TRAIN hazard ratios, `S(t|x) = exp(−H₀(t)·HR(x))`. **XGB `survival:aft`** →
analytic `S(t) = 1 − F_dist((log t − Xβ)/σ)` for the fitted distribution (`normal` → Φ,
`logistic` → sigmoid, `extreme` → `exp(−exp(·))`), with `Xβ = log(model.predict)`. Median
service day = first grid day with `S ≤ 0.5` (else *not reached*). Every model is checked for
`P30 ≤ P60 ≤ P90 ≤ P120` and `0 ≤ p ≤ 1`.

In [7]:
def _mtimes(model):
    for a in ("event_times_", "unique_times_"):
        if hasattr(model, a):
            return np.asarray(getattr(model, a))
    raise AttributeError("no model time attribute")

def cox_surv_matrix(model, X, times):
    arr = model.predict_survival_function(X, return_array=True)
    mts = _mtimes(model)
    out = np.empty((arr.shape[0], len(times)))
    for j, t in enumerate(times):
        k = int(np.searchsorted(mts, t, side="right") - 1)
        out[:, j] = arr[:, k] if k >= 0 else 1.0
    return np.clip(out, 0.0, 1.0)

def breslow_baseline(hr_train, y_tr, grid):
    """Breslow cumulative baseline hazard H0(grid) from XGB survival:cox hazard ratios on TRAIN."""
    order = np.argsort(y_tr["time"])
    t_s, e_s, hr_s = y_tr["time"][order], y_tr["event"][order], hr_train[order]
    risk_tail = np.cumsum(hr_s[::-1])[::-1]           # sum of HR for subjects still at risk
    ev_t = t_s[e_s]
    if len(ev_t) == 0:
        return np.zeros_like(grid, dtype=float)
    dH = 1.0 / np.maximum(risk_tail[e_s], 1e-12)
    uniq_t, inv = np.unique(ev_t, return_inverse=True)
    inv = np.asarray(inv).ravel()
    dH_by_t = np.zeros(len(uniq_t)); np.add.at(dH_by_t, inv, dH)
    H0_at = np.cumsum(dH_by_t)
    idx = np.searchsorted(uniq_t, grid, side="right") - 1
    return np.where(idx >= 0, H0_at[np.clip(idx, 0, len(H0_at) - 1)], 0.0)

def xgbcox_surv(hr, H0_grid, times):
    S = np.exp(-np.outer(hr, H0_grid))                # (n, len(grid)); grid == TGRID
    cols = [t - 1 for t in times]
    return np.clip(S[:, cols], 0.0, 1.0)

def aft_surv(xbeta, sigma, dist, times):
    z = (np.log(np.asarray(times, float))[None, :] - xbeta[:, None]) / sigma
    if dist == "normal":
        from scipy.stats import norm
        S = 1.0 - norm.cdf(z)
    elif dist == "logistic":
        S = 1.0 / (1.0 + np.exp(z))
    else:  # extreme  (Gumbel-min):  S(t) = exp(-exp(z))
        S = np.exp(-np.exp(np.clip(z, -30, 30)))
    return np.clip(S, 0.0, 1.0)

def median_from_grid(S_grid):
    below = S_grid <= 0.5
    med = np.full(S_grid.shape[0], np.nan)
    has = below.any(axis=1)
    med[has] = TGRID[below.argmax(axis=1)[has]]
    return med

## 7 · Temporal cross-validation
`N_FOLDS` expanding windows over TRAIN sorted by `snapshot_at` (no shuffle). Each row keeps
its authoritative admin-censored `(duration, event)` — the global cutoff is after every TRAIN
snapshot, so reusing the labels on a later time-slice introduces no future leakage.
`cv_composite()` is the Optuna objective (mean composite across folds).

In [8]:
_tr_ord = np.argsort(SNAP["TRAIN"], kind="stable")
_edges = [(0.55, 0.70), (0.70, 0.85), (0.85, 1.00)][:N_FOLDS]
FOLDS = []
n_tr = len(_tr_ord)
for lo, hi in _edges:
    tr_idx = _tr_ord[:int(lo * n_tr)]
    va_idx = _tr_ord[int(lo * n_tr):int(hi * n_tr)]
    FOLDS.append((np.sort(tr_idx), np.sort(va_idx)))
print("temporal folds (expanding, TRAIN only, sorted by snapshot_at, no shuffle):",
      [(len(a), len(b)) for a, b in FOLDS])
# Each row keeps its authoritative (duration, event) admin-censored at the GLOBAL cutoff (which is
# after every TRAIN snapshot), so using them as-is on a later time-slice introduces no future leakage.
qa("temporal_cv_no_leakage", "expanding window, authoritative admin-censored labels reused", "yes", True)

def cv_composite(make_S_fn):
    """make_S_fn(X_tr, y_tr, X_va) -> S_va (n, len(HORIZONS)). Returns mean composite over folds."""
    sc = []
    for tr_idx, va_idx in FOLDS:
        Xt, Xv = Xb["TRAIN"][tr_idx], Xb["TRAIN"][va_idx]
        yt = Surv.from_arrays(event=Y["TRAIN"]["event"][tr_idx], time=Y["TRAIN"]["time"][tr_idx])
        yv = Surv.from_arrays(event=Y["TRAIN"]["event"][va_idx], time=Y["TRAIN"]["time"][va_idx])
        try:
            S = np.clip(make_S_fn(Xt, yt, Xv), 0.0, 1.0)
        except Exception as e:
            print("   cv fold failed:", e); return 1.0
        risk = 1 - S[:, HORIZONS.index(90)]
        row = {}
        try:
            row["ipcw_c_index"] = float(concordance_index_ipcw(yt, yv, risk, tau=max(HORIZONS))[0])
        except Exception:
            row["ipcw_c_index"] = 0.5
        tt = [t for t in HORIZONS if t < yv["time"][yv["event"]].max() and (yv["time"] > t).sum() >= 20]
        if len(tt) > 1:
            cols = [HORIZONS.index(t) for t in tt]
            try:
                _, bs = brier_score(yt, yv, S[:, cols], tt)
                row["brier_90"] = float(bs[tt.index(90)]) if 90 in tt else float(np.mean(bs))
                row["ibs"] = float(integrated_brier_score(yt, yv, S[:, cols], tt))
            except Exception:
                row["brier_90"] = row["ibs"] = 0.2
        else:
            row["brier_90"] = row["ibs"] = 0.2
        row["cal_error_90"], _ = cal_error(risk, yv["time"], yv["event"].astype(int), 90)
        sc.append(composite(row))
    return float(np.mean(sc))

temporal folds (expanding, TRAIN only, sorted by snapshot_at, no shuffle): [(17711, 4831), (22542, 4830), (27372, 4831)]
  [PASS] temporal_cv_no_leakage: expanding window, authoritative admin-censored labels reused (exp yes) 


## 8 · Cox baseline reference
Reload `models/v2_cox_baseline.joblib` + `v2_survival_preprocessor.joblib` (nb14) and evaluate
on VALIDATION / TEST with the **same** metric helpers, so *baseline vs advanced* is
apples-to-apples. The advanced champion must beat this to be selected.

In [9]:
COX_BASE = joblib.load(MODELS / "v2_cox_baseline.joblib")
PRE_BASE = joblib.load(MODELS / "v2_survival_preprocessor.joblib")
_bcfg = json.loads((MODELS / "v2_baseline_config.json").read_text())
_bfs = _bcfg.get("feature_set", "SET_A_COMPACT")
_bcols = [c for c in FEATURE_SETS.get(_bfs, COMPACT) if c in FEATURES]
Xbase = {s: PRE_BASE.transform(mt.loc[masks[s], _bcols].astype({c: str for c in _bcols if c in CAT_FEATURES}))
             .astype(np.float64) for s in masks}
BASE_S = {s: cox_surv_matrix(COX_BASE, Xbase[s], ALL_H) for s in ("VALIDATION", "TEST")}
BASE_VAL = evaluate("COX_BASELINE", BASE_S["VALIDATION"], Y["TRAIN"], Y["VALIDATION"], DUR["VALIDATION"], EV["VALIDATION"])
print("baseline Cox VALIDATION:", {k: round(BASE_VAL[k], 4) for k in
      ("ipcw_c_index", "ibs", "brier_90", "cal_error_90")})

baseline Cox VALIDATION: {'ipcw_c_index': 0.9092, 'ibs': 0.0546, 'brier_90': 0.0614, 'cal_error_90': 0.1036}


## 9 · CoxNet
`CoxnetSurvivalAnalysis`, `l1_ratio ∈ {0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0}` with an
`alpha_min_ratio=0.01` path; the alpha is picked on the VALIDATION composite. Convergence
warnings are recorded, not suppressed. `v2_optuna_coxnet_trials.csv`.

In [10]:
L1_GRID = [0.1, 0.5, 0.9] if FAST_MODE else [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
CN_NALPHA = 12 if FAST_MODE else 30
cox_rows = []; _cn_warn = []
best_cn = None; best_cn_key = None; best_cn_sc = np.inf
_yv, _ytr = Y["VALIDATION"], Y["TRAIN"]
_hc = [ALL_H.index(h) for h in HORIZONS]
def _cn_cheap(Sv):
    """Cheap alpha-selection score on VALIDATION: 0.5·IBS + 0.3·Brier@90 + 0.2·(1−IPCW-C). Calibration is
    handled later by isotonic; AUC/cal_error skipped here to keep the alpha sweep fast."""
    risk = 1 - Sv[:, ALL_H.index(90)]
    try:
        ic = float(concordance_index_ipcw(_ytr, _yv, risk, tau=max(HORIZONS))[0])
    except Exception:
        ic = 0.5
    try:
        _, bs = brier_score(_ytr, _yv, Sv[:, _hc], HORIZONS)
        ibs = float(integrated_brier_score(_ytr, _yv, Sv[:, _hc], HORIZONS)); b90 = float(bs[2])
    except Exception:
        ibs = b90 = 0.2
    return 0.5 * ibs + 0.3 * b90 + 0.2 * (1 - ic), ic, ibs
for l1 in L1_GRID:
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        try:
            m = CoxnetSurvivalAnalysis(l1_ratio=l1, alpha_min_ratio=0.01, n_alphas=CN_NALPHA,
                                       max_iter=10**5, fit_baseline_model=True).fit(Xb["TRAIN"], Y["TRAIN"])
        except Exception as e:
            cox_rows.append({"l1_ratio": l1, "n_alphas": 0, "val_composite": np.nan, "note": f"fail:{e}"[:60]})
            continue
        _cn_warn += [str(wi.message)[:80] for wi in w if issubclass(wi.category, Warning)]
    alphas = m.alphas_
    va_sc = []
    for a in alphas[::max(1, len(alphas) // 8)]:
        try:
            arr = m.predict_survival_function(Xb["VALIDATION"], alpha=a, return_array=True)
            mts = _mtimes(m)
            Sv = np.clip(np.column_stack([arr[:, max(0, int(np.searchsorted(mts, t, "right") - 1))] for t in ALL_H]), 0, 1)
        except Exception:
            continue
        sc_a, ic_a, ibs_a = _cn_cheap(Sv)
        va_sc.append((sc_a, a, ic_a, ibs_a))
    if not va_sc:
        continue
    va_sc.sort(key=lambda x: x[0])
    sc, a_best, ic_best, ibs_best = va_sc[0]
    nz = int(np.sum(np.abs(m.coef_[:, list(alphas).index(a_best)]) > 1e-8)) if a_best in alphas else -1
    cox_rows.append({"l1_ratio": l1, "n_alphas": len(alphas), "alpha_selected": round(float(a_best), 5),
                     "nonzero_coef": nz, "val_ipcw_c": round(ic_best, 4),
                     "val_ibs": round(ibs_best, 4), "val_composite": round(sc, 5), "note": ""})
    if sc < best_cn_sc:
        best_cn_sc, best_cn, best_cn_key = sc, (m, a_best), (l1, a_best)
COXNET_TRIALS = pd.DataFrame(cox_rows)
COXNET_TRIALS.to_csv(TABLES / "v2_optuna_coxnet_trials.csv", index=False, encoding="utf-8-sig")
print(COXNET_TRIALS.to_string(index=False))
if _cn_warn:
    print("  CoxNet warnings (not suppressed):", sorted(set(_cn_warn))[:4])
CN_MODEL, CN_ALPHA = best_cn
def coxnet_S(X, times):
    arr = CN_MODEL.predict_survival_function(X, alpha=CN_ALPHA, return_array=True)
    mts = _mtimes(CN_MODEL)
    return np.clip(np.column_stack([arr[:, max(0, int(np.searchsorted(mts, t, "right") - 1))] for t in times]), 0, 1)
print(f"CoxNet selected: l1_ratio={best_cn_key[0]} alpha={best_cn_key[1]:.5g} | val composite {best_cn_sc:.5f}")

 l1_ratio  n_alphas  alpha_selected  nonzero_coef  val_ipcw_c  val_ibs  val_composite note
     0.05        30         0.15240            78      0.9256   0.0466        0.05392     
     0.10        30         0.07620            75      0.9254   0.0461        0.05347     
     0.25        30         0.03048            68      0.9247   0.0461        0.05356     
     0.50        30         0.01524            62      0.9241   0.0463        0.05382     
     0.75        30         0.01016            62      0.9238   0.0465        0.05397     
     0.90        30         0.00847            64      0.9237   0.0465        0.05403     
     1.00        30         0.00762            58      0.9236   0.0466        0.05407     
CoxNet selected: l1_ratio=0.1 alpha=0.076198 | val composite 0.05347


## 10 · XGBoost `survival:cox`
**Target encoding:** signed time — positive = observed event, negative = right-censored;
QA asserts the observed / censored counts are unchanged. Optuna (TPE + MedianPruner) over
`eta / max_depth / min_child_weight / subsample / colsample_bytree / gamma / reg_alpha /
reg_lambda / n_estimators`, objective = temporal-CV composite. Survival function via the
Breslow baseline hazard. `v2_optuna_xgb_cox_trials.csv`.

In [11]:
# --- XGB_COX_TARGET_ENCODING: survival:cox uses a SIGNED time label — positive = observed event,
# negative = right-censored. Counts must be unchanged by the encoding. ---
def cox_label(dur, ev):
    return np.where(ev == 1, dur, -dur).astype(float)
_lab_tr = cox_label(DUR["TRAIN"], EV["TRAIN"])
qa("xgb_cox_encoding_event_count", int((_lab_tr > 0).sum()), int(EV["TRAIN"].sum()),
   int((_lab_tr > 0).sum()) == int(EV["TRAIN"].sum()))
qa("xgb_cox_encoding_censored_count", int((_lab_tr < 0).sum()), int((EV["TRAIN"] == 0).sum()),
   int((_lab_tr < 0).sum()) == int((EV["TRAIN"] == 0).sum()))

def _xgb_cox_S(Xt, yt, Xv, params, rounds):
    dtr = xgb.DMatrix(Xt, label=cox_label(yt["time"], yt["event"].astype(int)))
    bst = xgb.train({**params, "objective": "survival:cox", "seed": SEED}, dtr, num_boost_round=rounds)
    hr_tr = bst.predict(dtr)
    H0 = breslow_baseline(hr_tr, yt, TGRID)
    hr_v = bst.predict(xgb.DMatrix(Xv))
    return bst, hr_v, H0

def xgb_cox_objective(trial):
    p = {"eta": trial.suggest_float("eta", 0.01, 0.3, log=True),
         "max_depth": trial.suggest_int("max_depth", 2, 6),
         "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 30.0, log=True),
         "subsample": trial.suggest_float("subsample", 0.6, 1.0),
         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
         "gamma": trial.suggest_float("gamma", 1e-3, 5.0, log=True),
         "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
         "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 20.0, log=True)}
    rounds = trial.suggest_int("n_estimators", 120, XGB_ROUNDS)
    def mk(Xt, yt, Xv):
        _, hr_v, H0 = _xgb_cox_S(Xt, yt, Xv, p, rounds)
        return xgbcox_surv(hr_v, H0, HORIZONS)
    return cv_composite(mk)

_t0 = time.time()
st_cox = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
                             pruner=optuna.pruners.MedianPruner(n_warmup_steps=3))
st_cox.optimize(xgb_cox_objective, n_trials=N_TRIALS, timeout=XGB_STUDY_TIMEOUT, show_progress_bar=False)
XGB_COX_NDONE = len([t for t in st_cox.trials if t.state.name == "COMPLETE"])
XGB_COX_TRIALS = st_cox.trials_dataframe()
XGB_COX_TRIALS.to_csv(TABLES / "v2_optuna_xgb_cox_trials.csv", index=False, encoding="utf-8-sig")
bp = st_cox.best_params
XGB_COX_PARAMS = {k: bp[k] for k in bp if k != "n_estimators"}
XGB_COX_ROUNDS = bp["n_estimators"]
XGB_COX_BST, _hrv, XGB_COX_H0 = _xgb_cox_S(Xb["TRAIN"], Y["TRAIN"], Xb["TRAIN"], XGB_COX_PARAMS, XGB_COX_ROUNDS)
def xgb_cox_S(X, times):
    return xgbcox_surv(XGB_COX_BST.predict(xgb.DMatrix(X)), XGB_COX_H0, times)
print(f"XGB survival:cox tuned in {time.time()-_t0:.0f}s | {XGB_COX_NDONE} complete trials | "
      f"best cv composite {st_cox.best_value:.5f} | rounds {XGB_COX_ROUNDS}")

  [PASS] xgb_cox_encoding_event_count: 25442 (exp 25442) 
  [PASS] xgb_cox_encoding_censored_count: 6761 (exp 6761) 


XGB survival:cox tuned in 816s | 120 complete trials | best cv composite 0.05014 | rounds 603


## 11 · XGBoost `survival:aft`
**Interval-censoring contract:** observed → `[t, t]`, right-censored → `[t, ∞)` via
`label_lower_bound` / `label_upper_bound`; QA asserts counts unchanged. Optuna also searches
`aft_loss_distribution ∈ {normal, logistic, extreme}` and `aft_loss_distribution_scale ∈
{0.5, 1, 1.5, 2, 3}`. Survival probabilities are the **documented analytic `S(t)`** of the
fitted distribution — not a bare median. `v2_optuna_xgb_aft_trials.csv`.

In [12]:
# --- AFT interval-censoring contract: observed -> [t, t]; right-censored -> [t, +inf). ---
def aft_bounds(dur, ev):
    lo = dur.astype(float).copy()
    up = np.where(ev == 1, dur, np.inf).astype(float)
    return lo, up
_lo_tr, _up_tr = aft_bounds(DUR["TRAIN"], EV["TRAIN"])
qa("xgb_aft_encoding_observed_count", int(np.isfinite(_up_tr).sum()), int(EV["TRAIN"].sum()),
   int(np.isfinite(_up_tr).sum()) == int(EV["TRAIN"].sum()))
qa("xgb_aft_encoding_censored_count", int(np.isinf(_up_tr).sum()), int((EV["TRAIN"] == 0).sum()),
   int(np.isinf(_up_tr).sum()) == int((EV["TRAIN"] == 0).sum()))

def _aft_dmat(X, lo=None, up=None):
    d = xgb.DMatrix(X)
    if lo is not None:
        d.set_float_info("label_lower_bound", lo); d.set_float_info("label_upper_bound", up)
    return d

def _xgb_aft_fit(Xt, yt, params, rounds, dist, scale):
    lo, up = aft_bounds(yt["time"], yt["event"].astype(int))
    bst = xgb.train({**params, "objective": "survival:aft", "eval_metric": "aft-nloglik",
                     "aft_loss_distribution": dist, "aft_loss_distribution_scale": scale, "seed": SEED},
                    _aft_dmat(Xt, lo, up), num_boost_round=rounds)
    return bst

def xgb_aft_objective(trial):
    dist = trial.suggest_categorical("dist", ["normal", "logistic", "extreme"])
    scale = trial.suggest_categorical("scale", [0.5, 1.0, 1.5, 2.0, 3.0])
    p = {"eta": trial.suggest_float("eta", 0.01, 0.3, log=True),
         "max_depth": trial.suggest_int("max_depth", 2, 6),
         "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 30.0, log=True),
         "subsample": trial.suggest_float("subsample", 0.6, 1.0),
         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
         "gamma": trial.suggest_float("gamma", 1e-3, 5.0, log=True),
         "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
         "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 20.0, log=True)}
    rounds = trial.suggest_int("n_estimators", 120, XGB_ROUNDS)
    def mk(Xt, yt, Xv):
        bst = _xgb_aft_fit(Xt, yt, p, rounds, dist, scale)
        xbeta = np.log(np.maximum(bst.predict(_aft_dmat(Xv)), 1e-6))
        return aft_surv(xbeta, scale, dist, HORIZONS)
    return cv_composite(mk)

_t0 = time.time()
st_aft = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
                             pruner=optuna.pruners.MedianPruner(n_warmup_steps=3))
st_aft.optimize(xgb_aft_objective, n_trials=N_TRIALS, timeout=XGB_STUDY_TIMEOUT, show_progress_bar=False)
XGB_AFT_NDONE = len([t for t in st_aft.trials if t.state.name == "COMPLETE"])
XGB_AFT_TRIALS = st_aft.trials_dataframe()
XGB_AFT_TRIALS.to_csv(TABLES / "v2_optuna_xgb_aft_trials.csv", index=False, encoding="utf-8-sig")
ap = st_aft.best_params
XGB_AFT_DIST, XGB_AFT_SCALE = ap["dist"], ap["scale"]
XGB_AFT_PARAMS = {k: ap[k] for k in ap if k not in ("n_estimators", "dist", "scale")}
XGB_AFT_ROUNDS = ap["n_estimators"]
XGB_AFT_BST = _xgb_aft_fit(Xb["TRAIN"], Y["TRAIN"], XGB_AFT_PARAMS, XGB_AFT_ROUNDS, XGB_AFT_DIST, XGB_AFT_SCALE)
def xgb_aft_S(X, times):
    xbeta = np.log(np.maximum(XGB_AFT_BST.predict(_aft_dmat(X)), 1e-6))
    return aft_surv(xbeta, XGB_AFT_SCALE, XGB_AFT_DIST, times)
AFT_PROB_OK = True   # analytic S(t) from the fitted AFT distribution — documented in surv_helpers
print(f"XGB survival:aft tuned in {time.time()-_t0:.0f}s | {XGB_AFT_NDONE} complete trials | "
      f"dist={XGB_AFT_DIST} scale={XGB_AFT_SCALE} | best cv composite {st_aft.best_value:.5f} | rounds {XGB_AFT_ROUNDS}")

  [PASS] xgb_aft_encoding_observed_count: 25442 (exp 25442) 
  [PASS] xgb_aft_encoding_censored_count: 6761 (exp 6761) 


XGB survival:aft tuned in 696s | 120 complete trials | dist=normal scale=0.5 | best cv composite 0.05684 | rounds 539


## 12 · RSF ranking diagnostic + GBS budget guard
RSF is fit on a stratified TRAIN subsample (sksurv RSF does not scale here — nb14).
`predict_survival_function` is benchmarked on 150 rows with a hard timeout; if it does not
return quickly, RSF is kept as a **ranking-only** diagnostic (IPCW-C / AUC), no probabilities.
GradientBoostingSurvival is attempted under a wall-clock budget → `FITTED` /
`SKIPPED_RUNTIME` / `FAILED`; the pipeline never blocks on it.

In [13]:
# RSF: benchmark predict_survival_function scalability before committing to probability metrics.
RSF_FIT_N = 8000 if FAST_MODE else 14000
_tr_pos = np.where(masks["TRAIN"])[0]
_ev_tr = EV["TRAIN"]
_r = np.random.RandomState(SEED)
_take = np.sort(np.concatenate([
    _r.choice(np.where(_ev_tr == 1)[0], int(RSF_FIT_N * _ev_tr.mean()), replace=False),
    _r.choice(np.where(_ev_tr == 0)[0], RSF_FIT_N - int(RSF_FIT_N * _ev_tr.mean()), replace=False)]))
Xrsf = Xb["TRAIN"][_take]
Yrsf = Surv.from_arrays(event=Y["TRAIN"]["event"][_take], time=Y["TRAIN"]["time"][_take])
rsf_cfg_rows = []; RSF_BEST = None; RSF_BEST_C = -1
for cfg in ([{"n_estimators": 25, "min_samples_leaf": 150, "max_depth": 6},
             {"n_estimators": 40, "min_samples_leaf": 80, "max_depth": 8}] if FAST_MODE else
            [{"n_estimators": 80, "min_samples_leaf": 60, "max_depth": 10},
             {"n_estimators": 120, "min_samples_leaf": 40, "max_depth": 12},
             {"n_estimators": 150, "min_samples_leaf": 25, "max_depth": 14}]):
    m = RandomSurvivalForest(**cfg, max_features="sqrt", max_samples=0.5, n_jobs=1, random_state=SEED).fit(Xrsf, Yrsf)
    c = float(concordance_index_censored(Y["VALIDATION"]["event"], Y["VALIDATION"]["time"],
                                         m.predict(Xb["VALIDATION"]))[0])
    rsf_cfg_rows.append({**cfg, "val_c_index": round(c, 4)})
    if c > RSF_BEST_C:
        RSF_BEST_C, RSF_BEST, RSF_CFG = c, m, cfg
pd.DataFrame(rsf_cfg_rows).to_csv(TABLES / "v2_optuna_rsf_trials.csv", index=False, encoding="utf-8-sig")
print("RSF configs:", rsf_cfg_rows)

RSF_PROB_OK = False
_bench = {}
if FAST_MODE:
    print("  RSF predict_survival_function benchmark skipped in FAST_MODE — nb14 established it does not "
          "scale on this dataset (>8 min / 600 rows). RSF kept as ranking diagnostic only. FULL run benchmarks it.")
else:
    _BENCH_N = 25   # tiny probe so a doomed call self-bounds instead of pinning a CPU for minutes
    def _try_surv():
        try:
            _t = time.time(); RSF_BEST.predict_survival_function(Xb["VALIDATION"][:_BENCH_N], return_array=True)
            _bench["s"] = time.time() - _t
        except Exception as e:
            _bench["err"] = str(e)[:80]
    th = threading.Thread(target=_try_surv, daemon=True); th.start(); th.join(RSF_BENCH_TIMEOUT)
    if th.is_alive() or "s" not in _bench:
        RSF_PROB_OK = False
        print(f"  RSF predict_survival_function did NOT return in {RSF_BENCH_TIMEOUT}s on {_BENCH_N} rows "
              f"-> RSF_PROBABILITY_SKIPPED_RUNTIME (ranking diagnostic only), consistent with nb14.")
    else:
        _extrap = _bench["s"] / _BENCH_N * masks["VALIDATION"].sum()
        RSF_PROB_OK = _extrap < 120.0
        print(f"  RSF predict_survival_function {_BENCH_N} rows in {_bench['s']:.1f}s "
              f"-> ~{_extrap:.0f}s extrapolated to full VALIDATION -> prob_ok={RSF_PROB_OK} "
              f"({'usable' if RSF_PROB_OK else 'RSF_PROBABILITY_SKIPPED_RUNTIME'})")
RSF_VAL_C = float(concordance_index_ipcw(Y["TRAIN"], Y["VALIDATION"], RSF_BEST.predict(Xb["VALIDATION"]),
                                         tau=max(HORIZONS))[0])
RSF_TEST_C = None   # filled after freeze

# GBS under a wall-clock budget guard
GBS_STATUS = "NOT_RUN"; GBS = None
if FAST_MODE:
    GBS_STATUS = "SKIPPED_RUNTIME"
    print("  GradientBoostingSurvival not attempted in FAST_MODE — nb14 established its fit does not "
          "complete in a practical time on full TRAIN. FULL run attempts it under a wall-clock budget guard.")
try:
    if FAST_MODE:
        raise RuntimeError("fast-mode skip")
    from sksurv.ensemble import GradientBoostingSurvivalAnalysis
    _res = {}
    def _fit_gbs():
        try:
            _res["m"] = GradientBoostingSurvivalAnalysis(
                n_estimators=(120 if FAST_MODE else 300), learning_rate=0.1,
                max_depth=3, subsample=0.7, random_state=SEED).fit(Xb["TRAIN"], Y["TRAIN"])
        except Exception as e:
            _res["err"] = str(e)[:100]
    g = threading.Thread(target=_fit_gbs, daemon=True); _t0 = time.time(); g.start(); g.join(GBS_BUDGET_S)
    if g.is_alive():
        GBS_STATUS = "SKIPPED_RUNTIME"
        print(f"  GradientBoostingSurvival exceeded the {GBS_BUDGET_S}s budget -> SKIPPED_RUNTIME")
    elif "m" in _res:
        GBS = _res["m"]; GBS_STATUS = "FITTED"
        print(f"  GradientBoostingSurvival fitted in {time.time()-_t0:.0f}s")
    else:
        GBS_STATUS = "FAILED"; print("  GBS failed:", _res.get("err"))
except Exception as e:
    if not FAST_MODE:
        GBS_STATUS = "UNAVAILABLE"; print("  GBS unavailable:", e)

RSF configs: [{'n_estimators': 80, 'min_samples_leaf': 60, 'max_depth': 10, 'val_c_index': 0.9227}, {'n_estimators': 120, 'min_samples_leaf': 40, 'max_depth': 12, 'val_c_index': 0.9242}, {'n_estimators': 150, 'min_samples_leaf': 25, 'max_depth': 14, 'val_c_index': 0.9259}]
  RSF predict_survival_function 25 rows in 0.1s -> ~25s extrapolated to full VALIDATION -> prob_ok=True (usable)


  GradientBoostingSurvival exceeded the 210s budget -> SKIPPED_RUNTIME


## 13 · Validation leaderboard → raw champion
All probability models on VALIDATION → `v2_advanced_validation_leaderboard.csv` with the
composite. RSF appears as a ranking-only row. The lowest-composite model is the **raw
champion**; the runner-up is also carried into calibration.

In [14]:
RAW_S = {"COX_BASELINE": {s: BASE_S[s] for s in ("VALIDATION", "TEST")}}
def _S_allh(fn, split):
    return np.clip(fn(Xb[split], ALL_H), 0.0, 1.0)
RAW_S["COXNET"] = {s: _S_allh(coxnet_S, s) for s in ("VALIDATION", "TEST")}
RAW_S["XGB_COX"] = {s: _S_allh(xgb_cox_S, s) for s in ("VALIDATION", "TEST")}
RAW_S["XGB_AFT"] = {s: _S_allh(xgb_aft_S, s) for s in ("VALIDATION", "TEST")}
RAW_S["KM_REFERENCE"] = {s: np.tile(KM_S_ALLH, (masks[s].sum(), 1)) for s in ("VALIDATION", "TEST")}
if GBS is not None:
    try:
        RAW_S["GBS"] = {s: cox_surv_matrix(GBS, Xb[s], ALL_H) for s in ("VALIDATION", "TEST")}
    except Exception as e:
        print("  GBS survival matrix failed:", e)

PROB_MODELS = [m for m in RAW_S]
VAL_ROWS = []
for m in PROB_MODELS:
    check_monotone(1 - RAW_S[m]["VALIDATION"], f"{m}_val_raw")
    r = evaluate(m, RAW_S[m]["VALIDATION"], Y["TRAIN"], Y["VALIDATION"], DUR["VALIDATION"], EV["VALIDATION"])
    r["calibration"] = "raw"; r["feature_set"] = BEST_FS if m not in ("KM_REFERENCE", "COX_BASELINE") else "-"
    r["composite"] = composite(r)
    VAL_ROWS.append(r)
# RSF ranking-only row (no probability -> Brier/IBS/cal NaN)
_rsf_val_rank = evaluate("RSF", 1 - np.tile([np.nan] * len(ALL_H), (masks["VALIDATION"].sum(), 1)),
                         Y["TRAIN"], Y["VALIDATION"], DUR["VALIDATION"], EV["VALIDATION"])
_rsf_val_rank["ipcw_c_index"] = RSF_VAL_C
_rsf_val_rank["c_index"] = float(concordance_index_censored(
    Y["VALIDATION"]["event"], Y["VALIDATION"]["time"], RSF_BEST.predict(Xb["VALIDATION"]))[0])
_rsf_val_rank["calibration"] = "raw"; _rsf_val_rank["feature_set"] = BEST_FS
_rsf_val_rank["composite"] = np.nan
VAL_ROWS.append(_rsf_val_rank)
VAL_LB = pd.DataFrame(VAL_ROWS)
_cols = ["model", "feature_set", "calibration", "ipcw_c_index", "ibs"] + [f"brier_{h}" for h in HORIZONS] \
        + [f"auc_{h}" for h in HORIZONS] + [f"cal_error_{h}" for h in HORIZONS] + ["composite"]
VAL_LB[[c for c in _cols if c in VAL_LB.columns]].to_csv(
    TABLES / "v2_advanced_validation_leaderboard.csv", index=False, encoding="utf-8-sig")
print(VAL_LB[["model", "ipcw_c_index", "ibs", "brier_90", "cal_error_90", "composite"]].round(4).to_string(index=False))

# COX_BASELINE and KM_REFERENCE are references, not advanced candidates — excluded from champion pool
_cand = VAL_LB[VAL_LB.composite.notna() & ~VAL_LB.model.isin(["KM_REFERENCE", "COX_BASELINE"])].sort_values("composite")
RAW_CHAMPION = _cand.iloc[0]["model"]
RAW_RUNNERUP = _cand.iloc[1]["model"]
print(f"RAW champion (VALIDATION composite): {RAW_CHAMPION} | runner-up: {RAW_RUNNERUP}")

  [PASS] prob_monotonic_COX_BASELINE_val_raw: 0 (exp 0) 
  [PASS] prob_bounds_COX_BASELINE_val_raw: 0 (exp 0) 


  [PASS] prob_monotonic_COXNET_val_raw: 0 (exp 0) 
  [PASS] prob_bounds_COXNET_val_raw: 0 (exp 0) 


  [PASS] prob_monotonic_XGB_COX_val_raw: 0 (exp 0) 
  [PASS] prob_bounds_XGB_COX_val_raw: 0 (exp 0) 


  [PASS] prob_monotonic_XGB_AFT_val_raw: 0 (exp 0) 
  [PASS] prob_bounds_XGB_AFT_val_raw: 0 (exp 0) 


  [PASS] prob_monotonic_KM_REFERENCE_val_raw: 0 (exp 0) 
  [PASS] prob_bounds_KM_REFERENCE_val_raw: 0 (exp 0) 


       model  ipcw_c_index    ibs  brier_90  cal_error_90  composite
COX_BASELINE        0.9092 0.0546    0.0614        0.1036     0.0709
      COXNET        0.9254 0.0461    0.0517        0.0954     0.0606
     XGB_COX        0.9143 0.0377    0.0443        0.0380     0.0490
     XGB_AFT        0.9163 0.0731    0.0936        0.2317     0.1041
KM_REFERENCE        0.5000 0.1226    0.1509           NaN     0.2092
         RSF        0.9273    NaN       NaN           NaN        NaN
RAW champion (VALIDATION composite): XGB_COX | runner-up: COXNET


## 14 · Calibration
Per-horizon **IPCW-weighted isotonic regression**, fit on VALIDATION only: censored-before-`h`
rows get weight 0, the rest are weighted `1/Ĝ(min(T,h))`. Cross-horizon monotonicity is
re-imposed by cumulative max. RAW vs CALIBRATED Brier / calibration-error →
`v2_advanced_raw_vs_calibrated.csv`. TEST reliability is never used to tune the maps.

In [15]:
# Per-horizon IPCW-weighted isotonic calibration, fit on VALIDATION only.
def ipcw_iso_fit(raw_risk, dur, ev, h):
    keep = ~((ev == 0) & (dur <= h))                       # drop censored-before-horizon (weight 0)
    y = ((dur <= h) & (ev == 1)).astype(float)[keep]
    w = np.where(dur[keep] <= h, 1.0 / np.array([G_hat(min(t, h)) for t in dur[keep]]),
                 1.0 / G_hat(h))
    iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
    iso.fit(raw_risk[keep], y, sample_weight=w)
    return iso

def apply_cal(cal_maps, S_allh):
    risk = 1 - S_allh
    out = risk.copy()
    for hi, h in enumerate(HORIZONS):
        out[:, ALL_H.index(h)] = cal_maps[h].predict(risk[:, ALL_H.index(h)])
    # enforce cross-horizon monotonicity (isotonic per-horizon can cross) — cumulative max over 30/60/90/120
    hc = [ALL_H.index(h) for h in HORIZONS]
    out[:, hc] = np.maximum.accumulate(out[:, hc], axis=1)
    return np.clip(1 - out, 0.0, 1.0)

CAL_TARGETS = [RAW_CHAMPION] + ([RAW_RUNNERUP] if RAW_RUNNERUP not in ("KM_REFERENCE",) else [])
CAL_MAPS = {}
rawcal_rows = []
for m in CAL_TARGETS:
    maps = {h: ipcw_iso_fit(1 - RAW_S[m]["VALIDATION"][:, ALL_H.index(h)],
                            DUR["VALIDATION"], EV["VALIDATION"], h) for h in HORIZONS}
    CAL_MAPS[m] = maps
    S_cal_val = apply_cal(maps, RAW_S[m]["VALIDATION"])
    check_monotone(1 - S_cal_val, f"{m}_val_cal")
    r_raw = VAL_LB[(VAL_LB.model == m) & (VAL_LB.calibration == "raw")].iloc[0]
    r_cal = evaluate(m, S_cal_val, Y["TRAIN"], Y["VALIDATION"], DUR["VALIDATION"], EV["VALIDATION"])
    for h in HORIZONS:
        rawcal_rows.append({"model": m, "horizon": h,
                            "raw_brier": round(r_raw[f"brier_{h}"], 4), "calibrated_brier": round(r_cal[f"brier_{h}"], 4),
                            "raw_cal_error": round(r_raw[f"cal_error_{h}"], 4),
                            "calibrated_cal_error": round(r_cal[f"cal_error_{h}"], 4),
                            "ranking_changed": False})
    r_cal.update({"calibration": "isotonic_ipcw", "feature_set": BEST_FS, "composite": composite(r_cal)})
    VAL_LB = pd.concat([VAL_LB, pd.DataFrame([r_cal])], ignore_index=True)
RAWCAL = pd.DataFrame(rawcal_rows)
RAWCAL.to_csv(TABLES / "v2_advanced_raw_vs_calibrated.csv", index=False, encoding="utf-8-sig")
print(RAWCAL.to_string(index=False))
VAL_LB[[c for c in _cols if c in VAL_LB.columns]].to_csv(
    TABLES / "v2_advanced_validation_leaderboard.csv", index=False, encoding="utf-8-sig")

_cand2 = VAL_LB[VAL_LB.composite.notna() & (VAL_LB.model != "KM_REFERENCE")].sort_values("composite")
CAL_CHAMPION_ROW = _cand2.iloc[0]
CAL_CHAMPION, CAL_METHOD = CAL_CHAMPION_ROW["model"], CAL_CHAMPION_ROW["calibration"]
print(f"best calibrated/raw model on VALIDATION composite: {CAL_CHAMPION} ({CAL_METHOD})")

  [PASS] prob_monotonic_XGB_COX_val_cal: 0 (exp 0) 
  [PASS] prob_bounds_XGB_COX_val_cal: 0 (exp 0) 


  [PASS] prob_monotonic_COXNET_val_cal: 0 (exp 0) 
  [PASS] prob_bounds_COXNET_val_cal: 0 (exp 0) 


  model  horizon  raw_brier  calibrated_brier  raw_cal_error  calibrated_cal_error  ranking_changed
XGB_COX       30     0.0174            0.0163         0.0011                0.0005            False
XGB_COX       60     0.0397            0.0378         0.0146                0.0061            False
XGB_COX       90     0.0443            0.0421         0.0380                0.0203            False
XGB_COX      120     0.0411            0.0387         0.0482                0.0463            False
 COXNET       30     0.0242            0.0194         0.0246                0.0007            False
 COXNET       60     0.0515            0.0425         0.0734                0.0069            False
 COXNET       90     0.0517            0.0428         0.0954                0.0223            False
 COXNET      120     0.0461            0.0391         0.0973                0.0456            False
best calibrated/raw model on VALIDATION composite: XGB_COX (isotonic_ipcw)


## 15 · Ensemble experiment
Top-2 by VALIDATION composite, coarse weight grid (0.0 … 1.0 step 0.1) on the survival
probabilities. Accepted **only** if IBS improves ≥1% and calibration@90 does not get worse;
otherwise the single model wins. `v2_advanced_ensemble_grid.csv`.

In [16]:
top2 = _cand2.drop_duplicates("model").head(2)["model"].tolist()
ENS_TESTED = len(top2) == 2 and "KM_REFERENCE" not in top2
ENS_SELECTED = False; ENS_W = None; ENS_MODELS = top2
best_ibs_single = _cand2.iloc[0]["ibs"]
if ENS_TESTED:
    def _best_S(m, split):
        use_cal = m in CAL_MAPS and _cand2[(_cand2.model == m)].iloc[0]["calibration"] != "raw"
        return apply_cal(CAL_MAPS[m], RAW_S[m][split]) if m in CAL_MAPS else RAW_S[m][split]
    Sa, Sb = _best_S(top2[0], "VALIDATION"), _best_S(top2[1], "VALIDATION")
    grid = []
    for w in np.round(np.arange(0, 1.0001, 0.1), 2):
        Sm = w * Sa + (1 - w) * Sb
        r = evaluate("ENS", Sm, Y["TRAIN"], Y["VALIDATION"], DUR["VALIDATION"], EV["VALIDATION"])
        grid.append({"w_%s" % top2[0]: w, "ibs": r["ibs"], "brier_90": r["brier_90"],
                     "cal_error_90": r["cal_error_90"], "composite": composite(r)})
    ENS_GRID = pd.DataFrame(grid)
    ENS_GRID.to_csv(TABLES / "v2_advanced_ensemble_grid.csv", index=False, encoding="utf-8-sig")
    best = ENS_GRID.sort_values("composite").iloc[0]
    ENS_W = float(best["w_%s" % top2[0]])
    _cal90_single = _cand2.iloc[0]["cal_error_90"]
    rel_ibs = (best_ibs_single - best["ibs"]) / best_ibs_single if np.isfinite(best_ibs_single) else 0
    ENS_SELECTED = bool(rel_ibs >= 0.01 and best["cal_error_90"] <= _cal90_single + 1e-3
                        and 0.0 < ENS_W < 1.0)
    print(f"ensemble {top2} best w={ENS_W} rel IBS gain {rel_ibs:+.1%} -> selected={ENS_SELECTED}")
else:
    print("ensemble not tested (need 2 distinct probability models)")

ensemble ['XGB_COX', 'COXNET'] best w=0.6 rel IBS gain +1.7% -> selected=True


## 16 · Freeze + TEST (opened once)
Freeze {model family, feature set, hyper-parameters, calibration method, ensemble weights,
horizons, risk thresholds}. The advanced champion is kept only if it beats the Cox baseline
on VALIDATION (≥5% IBS or Brier@90, or calibration MODERATE→GOOD) — else **COX_BASELINE stays
champion**. Then TEST is scored once for every frozen candidate →
`v2_advanced_test_leaderboard.csv`.

In [17]:
def champion_S(split):
    if ENS_SELECTED:
        def _S(m):
            return apply_cal(CAL_MAPS[m], RAW_S[m][split]) if m in CAL_MAPS else RAW_S[m][split]
        return np.clip(ENS_W * _S(ENS_MODELS[0]) + (1 - ENS_W) * _S(ENS_MODELS[1]), 0, 1)
    if CAL_CHAMPION in CAL_MAPS and CAL_METHOD != "raw":
        return apply_cal(CAL_MAPS[CAL_CHAMPION], RAW_S[CAL_CHAMPION][split])
    return RAW_S[CAL_CHAMPION][split]

FINAL_MODEL = ("ENSEMBLE[%s]" % "+".join(ENS_MODELS)) if ENS_SELECTED else CAL_CHAMPION
FINAL_CAL = "isotonic_ipcw" if (ENS_SELECTED or CAL_METHOD != "raw") else "none"
# advanced vs baseline acceptance is decided on VALIDATION, before TEST
_adv_val = evaluate(FINAL_MODEL, champion_S("VALIDATION"), Y["TRAIN"], Y["VALIDATION"], DUR["VALIDATION"], EV["VALIDATION"])
_rel_ibs_val = (BASE_VAL["ibs"] - _adv_val["ibs"]) / BASE_VAL["ibs"]
_rel_b90_val = (BASE_VAL["brier_90"] - _adv_val["brier_90"]) / BASE_VAL["brier_90"]
_cal_base_v = "GOOD" if BASE_VAL["cal_error_90"] < 0.03 else "MODERATE" if BASE_VAL["cal_error_90"] < 0.07 else "POOR"
_cal_adv_v = "GOOD" if _adv_val["cal_error_90"] < 0.03 else "MODERATE" if _adv_val["cal_error_90"] < 0.07 else "POOR"
ADV_BEATS_BASE = bool(_rel_ibs_val >= 0.05 or _rel_b90_val >= 0.05
                      or (_cal_base_v == "MODERATE" and _cal_adv_v == "GOOD"))
if not ADV_BEATS_BASE:
    FINAL_MODEL = "COX_BASELINE"; FINAL_CAL = "none"
    def champion_S(split):  # noqa: F811
        return BASE_S[split]
    _adv_val = evaluate(FINAL_MODEL, champion_S("VALIDATION"), Y["TRAIN"], Y["VALIDATION"], DUR["VALIDATION"], EV["VALIDATION"])
FROZEN = {"model_family": FINAL_MODEL, "feature_set": BEST_FS if FINAL_MODEL != "COX_BASELINE" else _bfs,
          "calibration_method": FINAL_CAL, "ensemble_weights": ({ENS_MODELS[0]: ENS_W, ENS_MODELS[1]: round(1 - ENS_W, 2)}
                                                               if ENS_SELECTED else None),
          "horizons": HORIZONS, "coxnet": {"l1_ratio": best_cn_key[0], "alpha": float(best_cn_key[1])},
          "xgb_cox": {**XGB_COX_PARAMS, "n_estimators": XGB_COX_ROUNDS},
          "xgb_aft": {**XGB_AFT_PARAMS, "n_estimators": XGB_AFT_ROUNDS, "dist": XGB_AFT_DIST, "scale": XGB_AFT_SCALE},
          "random_seed": SEED}
print(f"FROZEN: model={FINAL_MODEL} | calibration={FINAL_CAL} | adv_beats_baseline={ADV_BEATS_BASE}")

# ---- TEST opened once ----
CH_S = {"VALIDATION": champion_S("VALIDATION"), "TEST": champion_S("TEST")}
check_monotone(1 - CH_S["TEST"], "champion_test")
TEST_ROWS = []
_test_models = ["KM_REFERENCE", "COX_BASELINE", "COXNET", "XGB_COX", "XGB_AFT"] + (["GBS"] if "GBS" in RAW_S else [])
for m in _test_models:
    Sm = RAW_S[m]["TEST"]
    if m in CAL_MAPS and _cand2[_cand2.model == m].iloc[0]["calibration"] != "raw":
        Sm = apply_cal(CAL_MAPS[m], RAW_S[m]["TEST"])
    r = evaluate(m, Sm, Y["TRAIN"], Y["TEST"], DUR["TEST"], EV["TEST"])
    r["calibration"] = FINAL_CAL if m == CAL_CHAMPION else "raw"
    TEST_ROWS.append(r)
_ch_row = evaluate(FINAL_MODEL, CH_S["TEST"], Y["TRAIN"], Y["TEST"], DUR["TEST"], EV["TEST"])
_ch_row["calibration"] = FINAL_CAL
TEST_ROWS.append(_ch_row)
# RSF ranking-only on TEST
RSF_TEST_C = float(concordance_index_ipcw(Y["TRAIN"], Y["TEST"], RSF_BEST.predict(Xb["TEST"]), tau=max(HORIZONS))[0])
_rsf_t = {k: np.nan for k in TEST_ROWS[0]}
_rsf_t.update({"model": "RSF", "n": masks["TEST"].sum(), "events": int(EV["TEST"].sum()),
               "ipcw_c_index": RSF_TEST_C, "calibration": "raw",
               "c_index": float(concordance_index_censored(Y["TEST"]["event"], Y["TEST"]["time"],
                                                           RSF_BEST.predict(Xb["TEST"]))[0])})
TEST_ROWS.append(_rsf_t)
TEST_LB = pd.DataFrame(TEST_ROWS)
TEST_LB[[c for c in _cols if c in TEST_LB.columns]].to_csv(
    TABLES / "v2_advanced_test_leaderboard.csv", index=False, encoding="utf-8-sig")
print(TEST_LB[["model", "calibration", "ipcw_c_index", "ibs", "brier_90", "auc_90", "cal_error_90"]].round(4).to_string(index=False))
CH = TEST_LB[TEST_LB.model == FINAL_MODEL].iloc[-1]
BASE_T = TEST_LB[TEST_LB.model == "COX_BASELINE"].iloc[0]

FROZEN: model=ENSEMBLE[XGB_COX+COXNET] | calibration=isotonic_ipcw | adv_beats_baseline=True
  [PASS] prob_monotonic_champion_test: 0 (exp 0) 
  [PASS] prob_bounds_champion_test: 0 (exp 0) 


                   model   calibration  ipcw_c_index    ibs  brier_90  auc_90  cal_error_90
            KM_REFERENCE           raw        0.5000 0.1167    0.1352  0.5000           NaN
            COX_BASELINE           raw        0.8950 0.0570    0.0617  0.9360        0.0683
                  COXNET           raw        0.9073 0.0513    0.0536  0.9469        0.0728
                 XGB_COX isotonic_ipcw        0.9085 0.0507    0.0515  0.9489        0.0722
                 XGB_AFT           raw        0.9126 0.0475    0.0476  0.9551        0.0425
ENSEMBLE[XGB_COX+COXNET] isotonic_ipcw        0.9131 0.0488    0.0503  0.9507        0.0612
                     RSF           raw        0.9127    NaN       NaN     NaN           NaN


## 17 · TEST analysis
`v2_baseline_vs_advanced.csv`; horizon metrics; 10-bin champion calibration table + per-horizon
GOOD/MODERATE/POOR verdicts; risk groups (VALIDATION 90-day-risk terciles → TEST KM + HIGH-vs-LOW
log-rank); **motorcycle-grouped bootstrap CIs**; FIRST/LAST-episode diagnostics; segment
performance (history depth / brand `n≥50` / riding intensity / observed event type);
VALIDATION→TEST temporal degradation + verdict.

In [18]:
# baseline vs advanced
bva = []
for met in ["ipcw_c_index", "ibs", "brier_90", "auc_90", "cal_error_90"]:
    b, a = float(BASE_T[met]), float(CH[met])
    lower_better = met in ("ibs", "brier_90", "cal_error_90")
    delta = a - b
    rel = (b - a) / b if lower_better and b else (a - b) / b if b else np.nan
    verdict = ("better" if (delta < 0) == lower_better and abs(rel) >= 0.01 else
               "≈equal" if abs(rel) < 0.01 else "worse")
    bva.append({"metric": met, "cox_baseline": round(b, 4), "advanced_champion": round(a, 4),
                "absolute_delta": round(delta, 4), "relative_delta": round(rel, 4), "verdict": verdict})
BVA = pd.DataFrame(bva)
BVA.to_csv(TABLES / "v2_baseline_vs_advanced.csv", index=False, encoding="utf-8-sig")
print(BVA.to_string(index=False))

# horizon metrics + calibration verdicts
CAL_VERDICT = {}
for h in HORIZONS:
    ce = float(CH[f"cal_error_{h}"])
    CAL_VERDICT[h] = "GOOD" if ce < 0.03 else "MODERATE" if (ce < 0.07 or not np.isfinite(ce)) else "POOR"
    if not np.isfinite(ce): CAL_VERDICT[h] = "N/A"
hm = [{"model": r["model"], "horizon": h, "brier": r.get(f"brier_{h}"), "auc": r.get(f"auc_{h}"),
       "cal_error": r.get(f"cal_error_{h}"), "km_reference_risk": round(KM_RISK[h], 4)}
      for _, r in TEST_LB.iterrows() for h in ALL_H]
pd.DataFrame(hm).to_csv(TABLES / "v2_advanced_horizon_metrics.csv", index=False, encoding="utf-8-sig")

# calibration table (champion, TEST, 10 bins per horizon)
cal_tab_rows = []
for h in HORIZONS:
    _, rows = cal_error(1 - CH_S["TEST"][:, ALL_H.index(h)], DUR["TEST"], EV["TEST"], h)
    for rr in rows:
        cal_tab_rows.append({"model": FINAL_MODEL, **rr})
CAL_TAB = pd.DataFrame(cal_tab_rows)
CAL_TAB.to_csv(TABLES / "v2_advanced_calibration.csv", index=False, encoding="utf-8-sig")

# risk groups: VALIDATION 90d-risk terciles -> TEST
val_r90 = 1 - CH_S["VALIDATION"][:, ALL_H.index(90)]
test_r90 = 1 - CH_S["TEST"][:, ALL_H.index(90)]
q1, q2 = np.quantile(val_r90, [1 / 3, 2 / 3])
grp = np.where(test_r90 <= q1, "LOW", np.where(test_r90 <= q2, "MEDIUM", "HIGH"))
rg_rows = []
plt.figure(figsize=(8, 5))
for gname, col in zip(("LOW", "MEDIUM", "HIGH"), ("#2a9d8f", "#e9c46a", "#e76f51")):
    m = grp == gname
    kmf = KaplanMeierFitter().fit(DUR["TEST"][m], EV["TEST"][m], label=f"{gname} (n={m.sum()})")
    kmf.plot_survival_function(ci_show=False, color=col)
    med = float(kmf.median_survival_time_)
    rg_rows.append({"risk_group": gname, "n": int(m.sum()), "events": int(EV["TEST"][m].sum()),
                    "censored": int((EV["TEST"][m] == 0).sum()),
                    "event_rate_by_90d": round(float(1 - kmf.predict(90)), 4),
                    "km_median_days": round(med, 1) if np.isfinite(med) else np.nan})
plt.xlim(0, 365); plt.ylim(0, 1); plt.xlabel("days"); plt.ylabel("S(t)")
plt.title("12 · TEST risk-group Kaplan-Meier (thresholds from VALIDATION)"); savefig("12_risk_group_survival_curves.png")
RISK_GROUPS = pd.DataFrame(rg_rows)
RISK_GROUPS.to_csv(TABLES / "v2_advanced_risk_groups.csv", index=False, encoding="utf-8-sig")
lr = logrank_test(DUR["TEST"][grp == "HIGH"], DUR["TEST"][grp == "LOW"],
                  EV["TEST"][grp == "HIGH"], EV["TEST"][grp == "LOW"])
_rg = RISK_GROUPS.set_index("risk_group")
SEP_OK = bool(_rg.loc["HIGH", "event_rate_by_90d"] > _rg.loc["LOW", "event_rate_by_90d"] + 0.05)
print(RISK_GROUPS.to_string(index=False), f"\nHIGH vs LOW logrank p={lr.p_value:.2e} | separation_ok={SEP_OK}")

# grouped bootstrap CI (unit = motorcycle_id)
rng = np.random.RandomState(SEED)
te_moto = GROUPS["TEST"]; uniq = np.unique(te_moto)
moto_rows = {mm: np.where(te_moto == mm)[0] for mm in uniq}
hc = [ALL_H.index(h) for h in HORIZONS]
acc = {k: [] for k in ("ipcw_c_index", "ibs", "brier_90", "auc_90", "cal_error_90")}
S_ch_t = CH_S["TEST"]
for _ in range(N_BOOT):
    idx = np.concatenate([moto_rows[mm] for mm in rng.choice(uniq, len(uniq), replace=True)])
    yy = Surv.from_arrays(event=Y["TEST"]["event"][idx], time=Y["TEST"]["time"][idx])
    if yy["event"].sum() < 15:
        continue
    risk = 1 - S_ch_t[idx][:, ALL_H.index(120)]
    try: acc["ipcw_c_index"].append(float(concordance_index_ipcw(Y["TRAIN"], yy, risk, tau=max(HORIZONS))[0]))
    except Exception: pass
    tt = [t for t in HORIZONS if t < yy["time"][yy["event"]].max() and (yy["time"] > t).sum() >= 20]
    if len(tt) > 1:
        cc = [ALL_H.index(t) for t in tt]
        try:
            _, bs = brier_score(Y["TRAIN"], yy, S_ch_t[idx][:, cc], tt)
            if 90 in tt: acc["brier_90"].append(float(bs[tt.index(90)]))
            acc["ibs"].append(float(integrated_brier_score(Y["TRAIN"], yy, S_ch_t[idx][:, cc], tt)))
        except Exception: pass
    try:
        au, _ = cumulative_dynamic_auc(Y["TRAIN"], yy, risk, [90]); acc["auc_90"].append(float(np.atleast_1d(au)[0]))
    except Exception: pass
    ce, _ = cal_error(1 - S_ch_t[idx][:, ALL_H.index(90)], yy["time"], yy["event"].astype(int), 90)
    if np.isfinite(ce): acc["cal_error_90"].append(ce)
def _ci(a):
    a = np.array([x for x in a if np.isfinite(x)], float)
    return (round(float(a.mean()), 4), round(float(np.percentile(a, 2.5)), 4),
            round(float(np.percentile(a, 97.5)), 4)) if len(a) else (np.nan, np.nan, np.nan)
BOOT_CI = pd.DataFrame([{"metric": k, "estimate": _ci(v)[0], "ci_low": _ci(v)[1], "ci_high": _ci(v)[2],
                         "bootstrap_unit": "motorcycle_id", "n_replicates": N_BOOT} for k, v in acc.items()])
BOOT_CI.to_csv(TABLES / "v2_advanced_grouped_bootstrap_ci.csv", index=False, encoding="utf-8-sig")
print(BOOT_CI.to_string(index=False))

# FIRST / LAST episode
te_first = mt.loc[masks["TEST"], "_is_first_ep"].to_numpy()
te_last = mt.loc[masks["TEST"], "_is_last_ep"].to_numpy()
def _epi(subm, tag):
    yy = Surv.from_arrays(event=Y["TEST"]["event"][subm], time=Y["TEST"]["time"][subm])
    risk = 1 - S_ch_t[subm][:, ALL_H.index(120)]
    out = {"subset": tag, "n": int(subm.sum()), "events": int(yy["event"].sum()),
           "c_index": np.nan, "ipcw_c_index": np.nan, "ibs": np.nan, "brier_90": np.nan}
    if yy["event"].sum() >= 10:
        try: out["c_index"] = float(concordance_index_censored(yy["event"], yy["time"], risk)[0])
        except Exception: pass
        try: out["ipcw_c_index"] = float(concordance_index_ipcw(Y["TRAIN"], yy, risk, tau=max(HORIZONS))[0])
        except Exception: pass
        tt = [t for t in HORIZONS if t < yy["time"][yy["event"]].max() and (yy["time"] > t).sum() >= 20]
        if len(tt) > 1:
            cc = [ALL_H.index(t) for t in tt]
            try:
                _, bs = brier_score(Y["TRAIN"], yy, S_ch_t[subm][:, cc], tt)
                if 90 in tt: out["brier_90"] = float(bs[tt.index(90)])
                out["ibs"] = float(integrated_brier_score(Y["TRAIN"], yy, S_ch_t[subm][:, cc], tt))
            except Exception: pass
    return out
EPI = pd.DataFrame([{"subset": "FULL_TEST", "n": int(CH["n"]), "events": int(CH["events"]),
                     "c_index": float(CH["c_index"]), "ipcw_c_index": float(CH["ipcw_c_index"]),
                     "ibs": float(CH["ibs"]), "brier_90": float(CH["brier_90"])},
                    _epi(te_first, "FIRST_EPISODE_TEST"), _epi(te_last, "LAST_EPISODE_TEST")])
EPI.to_csv(TABLES / "v2_advanced_episode_diagnostic.csv", index=False, encoding="utf-8-sig")
print(EPI.round(4).to_string(index=False))
_gap = abs(EPI.set_index("subset").loc["FIRST_EPISODE_TEST", "c_index"] - EPI.set_index("subset").loc["FULL_TEST", "c_index"])
RECUR_CONCERN = "LOW" if (not np.isfinite(_gap) or _gap < 0.02) else "MODERATE" if _gap < 0.05 else "HIGH"

# segments
seg = mt.loc[masks["TEST"], ["brand", "riding_intensity", "previous_service_count", "next_event_type_audit"]].reset_index(drop=True)
seg["history_bin"] = pd.cut(seg.previous_service_count.fillna(0), [-1, 1, 3, np.inf], labels=["0-1", "2-3", "4+"])
seg["event_type"] = np.where(EV["TEST"] == 1, seg.next_event_type_audit.fillna("OTHER"), "CENSORED")
risk120 = 1 - S_ch_t[:, ALL_H.index(120)]
seg_rows = []
def seg_eval(col, min_n):
    for val in seg[col].dropna().unique():
        idx = np.where(seg[col].to_numpy() == val)[0]
        if len(idx) < min_n: continue
        yy = Surv.from_arrays(event=Y["TEST"]["event"][idx], time=Y["TEST"]["time"][idx])
        low = yy["event"].sum() < 10
        c = a90 = b90 = ce90 = np.nan
        if not low:
            try: c = float(concordance_index_ipcw(Y["TRAIN"], yy, risk120[idx], tau=max(HORIZONS))[0])
            except Exception: pass
            tt = [h for h in HORIZONS if h < yy["time"][yy["event"]].max() and (yy["time"] > h).sum() >= 20]
            if 90 in tt:
                cc = [ALL_H.index(t) for t in tt]
                try:
                    _, bs = brier_score(Y["TRAIN"], yy, S_ch_t[idx][:, cc], tt); b90 = float(bs[tt.index(90)])
                except Exception: pass
                try:
                    au, _ = cumulative_dynamic_auc(Y["TRAIN"], yy, risk120[idx], [90]); a90 = float(np.atleast_1d(au)[0])
                except Exception: pass
                ce90, _ = cal_error(1 - S_ch_t[idx][:, ALL_H.index(90)], yy["time"], yy["event"].astype(int), 90)
        seg_rows.append({"segment_type": col, "segment_value": str(val), "n": int(len(idx)),
                         "events": int(yy["event"].sum()), "censor_rate": round(1 - float(yy["event"].mean()), 3),
                         "ipcw_c": round(c, 4), "brier_90": round(b90, 4), "auc_90": round(a90, 4),
                         "calibration_error_90": round(ce90, 4), "note": "LOW_SAMPLE" if low else ""})
seg_eval("history_bin", 30); seg_eval("brand", 50); seg_eval("riding_intensity", 30); seg_eval("event_type", 30)
SEGPERF = pd.DataFrame(seg_rows)
SEGPERF.to_csv(TABLES / "v2_advanced_segment_performance.csv", index=False, encoding="utf-8-sig")
print(SEGPERF.to_string(index=False))
_hd = SEGPERF[(SEGPERF.segment_type == "history_bin") & SEGPERF.ipcw_c.notna()]
_br = SEGPERF[(SEGPERF.segment_type == "brand") & SEGPERF.ipcw_c.notna()]
BEST_HD = _hd.sort_values("ipcw_c").iloc[-1]["segment_value"] if len(_hd) else "n/a"
WORST_HD = _hd.sort_values("ipcw_c").iloc[0]["segment_value"] if len(_hd) else "n/a"
BEST_BR = _br.sort_values("ipcw_c").iloc[-1]["segment_value"] if len(_br) else "n/a"
WORST_BR = _br.sort_values("ipcw_c").iloc[0]["segment_value"] if len(_br) else "n/a"

# temporal generalization
TEMP = pd.DataFrame([{"metric": m, "validation": round(float(_adv_val[m]), 4), "test": round(float(CH[m]), 4),
                      "degradation": round(float(CH[m]) - float(_adv_val[m]), 4)}
                     for m in ("ipcw_c_index", "ibs", "brier_90", "auc_90", "cal_error_90")])
TEMP.to_csv(TABLES / "v2_advanced_temporal_generalization.csv", index=False, encoding="utf-8-sig")
print(TEMP.to_string(index=False))
_dc = float(_adv_val["ipcw_c_index"] - CH["ipcw_c_index"])
TEMP_VERDICT = ("STABLE" if _dc < 0.03 else "MODERATE DRIFT" if _dc < 0.07 else "TEMPORAL GENERALIZATION LIMITATION")

      metric  cox_baseline  advanced_champion  absolute_delta  relative_delta verdict
ipcw_c_index        0.8950             0.9131          0.0181          0.0203  better
         ibs        0.0570             0.0488         -0.0083          0.1449  better
    brier_90        0.0617             0.0503         -0.0114          0.1847  better
      auc_90        0.9360             0.9507          0.0147          0.0157  better
cal_error_90        0.0683             0.0612         -0.0071          0.1043  better


risk_group    n  events  censored  event_rate_by_90d  km_median_days
       LOW 2301     259      2042             0.0007           226.4
    MEDIUM 1228     427       801             0.1211           164.8
      HIGH  941     596       345             0.7090            63.5 
HIGH vs LOW logrank p=1.91e-303 | separation_ok=True


      metric  estimate  ci_low  ci_high bootstrap_unit  n_replicates
ipcw_c_index    0.9131  0.9052   0.9204  motorcycle_id           500
         ibs    0.0412  0.0379   0.0447  motorcycle_id           500
    brier_90    0.0501  0.0454   0.0547  motorcycle_id           500
      auc_90    0.9508  0.9439   0.9585  motorcycle_id           500
cal_error_90    0.0641  0.0499   0.0788  motorcycle_id           500
            subset    n  events  c_index  ipcw_c_index    ibs  brier_90
         FULL_TEST 4470    1282   0.8780        0.9131 0.0488    0.0503
FIRST_EPISODE_TEST  857     161   0.8826        0.9159 0.0215    0.0301
 LAST_EPISODE_TEST 3188       0      NaN           NaN    NaN       NaN


    segment_type segment_value    n  events  censor_rate  ipcw_c  brier_90  auc_90  calibration_error_90       note
     history_bin           2-3 1085     245        0.774  0.9501    0.0235  0.9853                0.1086           
     history_bin            4+ 1802     719        0.601  0.8498    0.0847  0.9009                0.0750           
     history_bin           0-1 1583     318        0.799  0.9342    0.0296  0.9640                0.0134           
           brand        Yamaha  404     127        0.686  0.8384    0.0747  0.8743                0.0761           
           brand       Mondial  805     351        0.564  0.8661    0.0837  0.9232                0.0869           
           brand           TVS  431      58        0.865  0.9829    0.0039  0.9979                0.0395           
           brand         Honda 1480     321        0.783  0.9349    0.0284  0.9674                0.0399           
           brand        CFMOTO  164      20        0.878     NaN    0.00

## 18 · Importance · CoxNet coefficients · latency · V1↔V2
Survival-appropriate feature importance for the champion's driving model (XGB gain or |coef|),
top 25 → `v2_advanced_feature_importance.csv`. CoxNet non-zero coefficients / hazard ratios →
`v2_coxnet_coefficients.csv`. Champion inference latency at 1 / 100 / 1000 predictions (CPU) →
production-feasibility flag. Secondary V1-days vs V2-median-days diagnostic on the observed
TEST subset (**not** used for selection).

In [19]:
# permutation importance (survival) for the final champion's driving model
imp_model_name = ("XGB_AFT" if "XGB_AFT" in FINAL_MODEL else "XGB_COX" if "XGB_COX" in FINAL_MODEL
                  else "COXNET" if "COXNET" in FINAL_MODEL else "COX_BASELINE")
try:
    if imp_model_name == "XGB_AFT":
        gain = XGB_AFT_BST.get_score(importance_type="gain")
    elif imp_model_name == "XGB_COX":
        gain = XGB_COX_BST.get_score(importance_type="gain")
    else:
        gain = {}
    if gain:
        gi = pd.DataFrame({"feature": [ENC_NAMES[int(k[1:])] if k.startswith("f") and k[1:].isdigit() else k
                                       for k in gain], "gain": list(gain.values())}).sort_values("gain", ascending=False)
    else:
        coef = np.ravel(CN_MODEL.coef_[:, list(CN_MODEL.alphas_).index(CN_ALPHA)]) if imp_model_name == "COXNET" \
               else np.ravel(COX_BASE.coef_)
        gi = pd.DataFrame({"feature": ENC_NAMES[:len(coef)], "gain": np.abs(coef)}).sort_values("gain", ascending=False)
    gi.head(25).to_csv(TABLES / "v2_advanced_feature_importance.csv", index=False, encoding="utf-8-sig")
    TOP_FEATS = gi.head(10).feature.tolist()
    _t = gi.head(20).iloc[::-1]
    plt.figure(figsize=(8, 7)); plt.barh(_t.feature, _t.gain, color="#3b6ea5")
    plt.title(f"16 · feature importance ({imp_model_name}, {'gain' if gain else '|coef|'})")
    savefig("16_feature_importance.png")
except Exception as e:
    TOP_FEATS = []; print("importance failed:", e)

# CoxNet coefficients / hazard ratios
try:
    cnc = np.ravel(CN_MODEL.coef_[:, list(CN_MODEL.alphas_).index(CN_ALPHA)])
    CN_HR = pd.DataFrame({"feature": ENC_NAMES[:len(cnc)], "coefficient": cnc, "hazard_ratio": np.exp(cnc)})
    CN_HR = CN_HR[CN_HR.coefficient.abs() > 1e-8].assign(a=lambda d: d.coefficient.abs()).sort_values("a", ascending=False)
    CN_HR.drop(columns="a").to_csv(TABLES / "v2_coxnet_coefficients.csv", index=False, encoding="utf-8-sig")
    print(f"CoxNet non-zero coefficients: {len(CN_HR)} / {len(cnc)}")
except Exception as e:
    CN_HR = pd.DataFrame(); print("CoxNet HR failed:", e)

# latency / production feasibility
_LATSRC = Xbase["TEST"] if FINAL_MODEL == "COX_BASELINE" else Xb["TEST"]
def _lat(n):
    X = _LATSRC[:n] if n <= len(_LATSRC) else np.repeat(_LATSRC, int(np.ceil(n / len(_LATSRC))), 0)[:n]
    _t = time.time(); champion_S_predict(X); return (time.time() - _t) * 1000
def champion_S_predict(X):
    if FINAL_MODEL == "COX_BASELINE":
        return cox_surv_matrix(COX_BASE, X, ALL_H)
    parts = []
    names = ENS_MODELS if ENS_SELECTED else [CAL_CHAMPION]
    ws = [ENS_W, 1 - ENS_W] if ENS_SELECTED else [1.0]
    for nm, w in zip(names, ws):
        fn = {"COXNET": coxnet_S, "XGB_COX": xgb_cox_S, "XGB_AFT": xgb_aft_S,
              "KM_REFERENCE": lambda X, t: np.tile(KM_S_ALLH, (len(X), 1))}[nm]
        S = fn(X, ALL_H)
        if nm in CAL_MAPS and FINAL_CAL != "none":
            S = apply_cal(CAL_MAPS[nm], S)
        parts.append(w * S)
    return np.clip(sum(parts), 0, 1)
LAT = {n: round(_lat(n), 2) for n in (1, 100, 1000)}
PROD_OK = LAT[1000] < 5000
RUNTIME = pd.DataFrame([{"model": "XGB_COX", "fit_seconds": None, "note": f"tuned {N_TRIALS} trials"},
                        {"model": "XGB_AFT", "fit_seconds": None, "note": f"tuned {N_TRIALS} trials"},
                        {"model": FINAL_MODEL, "predict_1_ms": LAT[1], "predict_100_ms": LAT[100],
                         "predict_1000_ms": LAT[1000], "production_inference_feasible": PROD_OK}])
RUNTIME.to_csv(TABLES / "v2_advanced_runtime.csv", index=False, encoding="utf-8-sig")
print("latency ms:", LAT, "| production feasible:", PROD_OK)

# V1 vs V2 observed-days diagnostic (secondary — NOT used for selection)
V1V2 = {}
try:
    v1p = pd.read_parquet(OUTPUTS / "v1_final_tuning_test_predictions.parquet")
    med_days = median_from_grid(champion_S_predict(Xb["TEST"]) if FINAL_MODEL != "COX_BASELINE"
                                else cox_surv_matrix(COX_BASE, Xbase["TEST"], ALL_H)) if False else None
except Exception:
    v1p = None
try:
    # champion median days on TEST grid
    if FINAL_MODEL == "COX_BASELINE":
        arr = COX_BASE.predict_survival_function(Xbase["TEST"], return_array=True); mts = _mtimes(COX_BASE)
        Sg = np.clip(np.column_stack([arr[:, max(0, int(np.searchsorted(mts, t, "right") - 1))] for t in TGRID]), 0, 1)
    elif "XGB_AFT" in FINAL_MODEL:
        xbeta = np.log(np.maximum(XGB_AFT_BST.predict(_aft_dmat(Xb["TEST"])), 1e-6))
        Sg = aft_surv(xbeta, XGB_AFT_SCALE, XGB_AFT_DIST, TGRID)
    elif "XGB_COX" in FINAL_MODEL:
        Sg = np.exp(-np.outer(XGB_COX_BST.predict(xgb.DMatrix(Xb["TEST"])), XGB_COX_H0))
    else:
        arr = CN_MODEL.predict_survival_function(Xb["TEST"], alpha=CN_ALPHA, return_array=True)
        mts = _mtimes(CN_MODEL)
        Sg = np.clip(np.column_stack([arr[:, max(0, int(np.searchsorted(mts, t, "right") - 1))] for t in TGRID]), 0, 1)
    CH_MED = median_from_grid(Sg)
except Exception as e:
    CH_MED = np.full(masks["TEST"].sum(), np.nan); print("median calc failed:", e)
if v1p is not None:
    obs = (EV["TEST"] == 1)
    j = mt.loc[masks["TEST"], "snapshot_id"].reset_index(drop=True)
    v1m = v1p.set_index("snapshot_id")["pred_next_service_days"] if "pred_next_service_days" in v1p.columns else None
    if v1m is not None:
        v1a = j.map(v1m).to_numpy()
        ok = obs & np.isfinite(v1a) & np.isfinite(CH_MED)
        if ok.sum() > 30:
            V1V2 = {"n": int(ok.sum()),
                    "v1_medae": round(float(np.median(np.abs(v1a[ok] - DUR["TEST"][ok]))), 1),
                    "v2_medae": round(float(np.median(np.abs(CH_MED[ok] - DUR["TEST"][ok]))), 1)}
    plt.figure(figsize=(6, 6))
    if V1V2:
        plt.scatter(DUR["TEST"][ok], CH_MED[ok], s=6, alpha=.3, label="V2 median")
        plt.plot([0, 300], [0, 300], "k--"); plt.xlabel("observed days"); plt.ylabel("V2 predicted median days")
        plt.title("20 · V1 vs V2 observed-days diagnostic"); savefig("20_v1_vs_v2_observed_days_diagnostic.png")
print("V1 vs V2 median-days (observed TEST subset):", V1V2 or "v1 predictions unavailable")

CoxNet non-zero coefficients: 75 / 243


latency ms: {1: 3.66, 100: 115.73, 1000: 1144.99} | production feasible: True
V1 vs V2 median-days (observed TEST subset): v1 predictions unavailable


## 19 · Artifacts · verdicts · QA · report
Save `v2_advanced_champion.joblib`, `v2_advanced_preprocessor.joblib`,
`v2_advanced_calibrator.joblib`, per-family models, `v2_advanced_config.json`;
`outputs/v2_advanced_test_predictions.parquet` (§56 columns). Reproducibility (XGB refit,
seed 42), full QA gate, figures, report, README line, non-breaking Control Center update
(real measured metrics only). Verdicts: **advanced-vs-baseline** (STRONG / MODERATE / SMALL /
NO improvement), **V2 signal** (STRONG / MODERATE / WEAK), **V2 status** (COMPLETE / NEEDS
CALIBRATION ROBUSTNESS / NEEDS MORE MODELING). Then the §83 report block.

In [20]:
# figures: leaderboards, baseline-vs-advanced, brier/auc by horizon, calibration, temporal, segments, optuna, examples
VAL_LB_P = VAL_LB[VAL_LB.model != "KM_REFERENCE"].drop_duplicates(["model", "calibration"])
for _df, _fid, _ttl in ((VAL_LB_P, "01", "validation"), (TEST_LB[TEST_LB.model != "RSF"], "02", "test")):
    d = _df.set_index(_df.model + "/" + _df.get("calibration", "raw").astype(str))[["ipcw_c_index", "ibs", "cal_error_90"]].astype(float)
    d.plot(kind="bar", figsize=(9, 4)); plt.title(f"{_fid} · {_ttl} leaderboard"); savefig(f"{_fid}_{_ttl}_leaderboard.png")
for _fid, met, ttl in (("03", "ibs", "IBS"), ("04", "ipcw_c_index", "IPCW C-index")):
    sub = BVA[BVA.metric == met]
    plt.figure(figsize=(5, 4)); plt.bar(["Cox baseline", "advanced"], [sub.cox_baseline.iloc[0], sub.advanced_champion.iloc[0]],
                                        color=["#264653", "#e76f51"])
    plt.title(f"{_fid} · baseline vs advanced — {ttl} (TEST)"); savefig(f"{_fid}_baseline_vs_advanced_{'ibs' if met=='ibs' else 'cindex'}.png")
_hm = pd.DataFrame(hm)
_hm[_hm.model.isin([FINAL_MODEL, "COX_BASELINE", "KM_REFERENCE"])].pivot_table(index="horizon", columns="model", values="brier").plot(
    kind="bar", figsize=(8, 4)); plt.title("05 · Brier by horizon (TEST)"); savefig("05_brier_by_horizon.png")
_hm[_hm.model.isin([FINAL_MODEL, "COX_BASELINE", "KM_REFERENCE"])].pivot_table(index="horizon", columns="model", values="auc").plot(
    kind="bar", figsize=(8, 4)); plt.axhline(0.5, color="#999", ls=":"); plt.title("06 · time-dependent AUC by horizon (TEST)")
savefig("06_auc_by_horizon.png")
for hi, h in enumerate(HORIZONS):
    d = CAL_TAB[CAL_TAB.horizon == h]
    plt.figure(figsize=(5, 5)); plt.plot([0, 1], [0, 1], "k--", lw=1)
    if len(d): plt.plot(d.mean_predicted, d.observed_km, "o-", color="#e76f51")
    plt.xlim(0, 1); plt.ylim(0, 1); plt.xlabel("predicted risk"); plt.ylabel("observed (KM) risk")
    plt.title(f"0{7+hi} · calibration @{h}d ({FINAL_MODEL})"); savefig(f"0{7+hi}_calibration_{h}d.png")
if len(CAL_TARGETS):
    m0 = CAL_TARGETS[0]; d = RAWCAL[RAWCAL.model == m0]
    plt.figure(figsize=(6, 4)); x = np.arange(len(d))
    plt.bar(x - 0.2, d.raw_cal_error, 0.4, label="raw"); plt.bar(x + 0.2, d.calibrated_cal_error, 0.4, label="calibrated")
    plt.xticks(x, d.horizon); plt.legend(); plt.title(f"11 · raw vs calibrated cal-error ({m0})")
    savefig("11_raw_vs_calibrated_90d.png")
plt.figure(figsize=(7, 4)); plt.bar(TEMP.metric, TEMP.degradation.abs(), color="#457b9d")
plt.title("13 · |validation → test degradation|"); plt.xticks(rotation=30, ha="right"); savefig("13_temporal_generalization.png")
if len(_hd):
    plt.figure(figsize=(6, 4)); plt.bar(_hd.segment_value, _hd.ipcw_c, color="#264653"); plt.ylim(0.4, 1.0)
    plt.title("14 · IPCW-C by history depth (TEST)"); savefig("14_performance_by_history_depth.png")
if len(_br):
    _brs = _br.sort_values("ipcw_c"); plt.figure(figsize=(8, 4)); plt.barh(_brs.segment_value, _brs.ipcw_c, color="#2a9d8f")
    plt.xlim(0.4, 1.0); plt.title("15 · IPCW-C by brand (TEST, n>=50)"); savefig("15_performance_by_brand.png")
for _fid, st_, nm in (("18", st_aft, "xgb_aft"), ("19", st_cox, "xgb_cox")):
    try:
        vals = [t.value for t in st_.trials if t.value is not None]
        plt.figure(figsize=(7, 4)); plt.plot(vals, "o-", ms=3); plt.plot(np.minimum.accumulate(vals), color="#e76f51")
        plt.xlabel("trial"); plt.ylabel("cv composite"); plt.title(f"{_fid} · Optuna history {nm}"); savefig(f"{_fid}_optuna_history_{nm}.png")
    except Exception: pass
# example survival curves — deterministic quantile picks
order = np.argsort(test_r90)
picks = {"LOW": order[int(.10 * len(order))], "MED": order[int(.50 * len(order))], "HIGH": order[int(.90 * len(order))]}
plt.figure(figsize=(8, 5))
for (lbl, i), c in zip(picks.items(), ("#2a9d8f", "#e9c46a", "#e76f51")):
    if FINAL_MODEL == "COX_BASELINE":
        arr = COX_BASE.predict_survival_function(Xbase["TEST"][[i]], return_array=True)[0]; mts = _mtimes(COX_BASE)
        plt.step(mts, arr, where="post", color=c, label=f"{lbl} · {'event' if EV['TEST'][i] else 'censored'} d{DUR['TEST'][i]:.0f}")
    else:
        Sg_i = champion_S_predict(Xb["TEST"][[i]])
        # rough curve from ALL_H points
        plt.plot([0] + ALL_H, [1] + list(Sg_i[0]), "o-", color=c,
                 label=f"{lbl} · {'event' if EV['TEST'][i] else 'censored'} d{DUR['TEST'][i]:.0f}")
    plt.axvline(DUR["TEST"][i], color=c, ls=":", alpha=.5)
plt.xlim(0, 365); plt.ylim(0, 1); plt.legend(); plt.xlabel("days"); plt.ylabel("S(t)")
plt.title(f"17 · example {FINAL_MODEL} survival curves (TEST, quantile-picked)"); savefig("17_example_survival_curves.png")

# example prediction table (deterministic quantiles)
ex_rows = []
_KMCOL = next((c for c in ("recent_90d_km", "initial_mileage_km", "annual_km_baseline", "recent_180d_km")
               if c in mt.columns), None)
_meta_cols = ["brand", "category", "previous_service_count"] + ([_KMCOL] if _KMCOL else [])
_meta = mt.loc[masks["TEST"], _meta_cols].reset_index(drop=True)
for band, frac in [("LOW", .10), ("LOW", .20), ("MEDIUM", .45), ("MEDIUM", .55), ("HIGH", .85), ("HIGH", .93)]:
    i = int(order[int(frac * len(order))])
    r = CH_S["TEST"][i]
    ex_rows.append({"risk_band": band, "brand": _meta.brand.iloc[i], "category": _meta.category.iloc[i],
                    f"{_KMCOL or 'km'}": (round(float(_meta[_KMCOL].iloc[i]), 0) if _KMCOL and np.isfinite(_meta[_KMCOL].iloc[i]) else None),
                    "prior_services": int(_meta.previous_service_count.iloc[i] or 0),
                    "P30": round(float(1 - r[0]), 3), "P60": round(float(1 - r[1]), 3),
                    "P90": round(float(1 - r[2]), 3), "P120": round(float(1 - r[3]), 3),
                    "median_days": None if not np.isfinite(CH_MED[i]) else int(CH_MED[i]),
                    "outcome": "event d%.0f" % DUR["TEST"][i] if EV["TEST"][i] else "censored d%.0f" % DUR["TEST"][i]})
EXAMPLES = pd.DataFrame(ex_rows)
EXAMPLES.to_csv(TABLES / "v2_advanced_example_predictions.csv", index=False, encoding="utf-8-sig")
print(EXAMPLES.to_string(index=False))

# ---- FAST reference (captured before this run overwrites the config) ----
FAST_REF = {}
_prev_cfg = MODELS / "v2_advanced_config.json"
if not FAST_MODE and _prev_cfg.exists():
    try:
        _pc = json.loads(_prev_cfg.read_text())
        if _pc.get("fast_mode") or _pc.get("run_mode") == "FAST" or _pc.get("status") == "PROVISIONAL":
            FAST_REF = _pc.get("test_metrics", {}) or {}
            (TABLES / "v2_advanced_fast_reference.json").write_text(json.dumps(FAST_REF, indent=2))
    except Exception as e:
        print("FAST reference capture skipped:", e)
elif not FAST_MODE and (TABLES / "v2_advanced_fast_reference.json").exists():
    try:
        FAST_REF = json.loads((TABLES / "v2_advanced_fast_reference.json").read_text())
    except Exception:
        FAST_REF = {}

# ---- artifacts ----
if FINAL_MODEL == "COX_BASELINE":
    joblib.dump(COX_BASE, MODELS / "v2_advanced_champion.joblib")
    joblib.dump(PRE_BASE, MODELS / "v2_advanced_preprocessor.joblib")
else:
    champ_payload = {"final_model": FINAL_MODEL, "calibration": FINAL_CAL, "ens_selected": ENS_SELECTED,
                     "ens_models": ENS_MODELS, "ens_w": ENS_W, "cal_champion": CAL_CHAMPION,
                     "coxnet": CN_MODEL, "coxnet_alpha": CN_ALPHA,
                     "xgb_cox_model": XGB_COX_BST, "xgb_cox_H0": XGB_COX_H0,
                     "xgb_aft_model": XGB_AFT_BST, "xgb_aft_dist": XGB_AFT_DIST, "xgb_aft_scale": XGB_AFT_SCALE,
                     "enc_names": ENC_NAMES, "all_h": ALL_H}
    joblib.dump(champ_payload, MODELS / "v2_advanced_champion.joblib")
    joblib.dump(PRE, MODELS / "v2_advanced_preprocessor.joblib")
joblib.dump(CAL_MAPS.get(CAL_CHAMPION, {}), MODELS / "v2_advanced_calibrator.joblib")
joblib.dump(CN_MODEL, MODELS / "v2_coxnet_model.joblib")
XGB_COX_BST.save_model(str(MODELS / "v2_xgb_cox_model.json"))
XGB_AFT_BST.save_model(str(MODELS / "v2_xgb_aft_model.json"))

CONFIG = {"dataset_version": DATASET_VERSION, "notebook": "15_v2_survival_advanced", "fast_mode": FAST_MODE,
          "run_mode": RUN_MODE, "status": "FINAL" if not FAST_MODE else "PROVISIONAL",
          "optuna_trials": {"coxnet_grid": int(len(COXNET_TRIALS)), "xgb_cox_complete": int(XGB_COX_NDONE),
                            "xgb_aft_complete": int(XGB_AFT_NDONE), "target_per_model": N_TRIALS},
          "temporal_folds": int(len(FOLDS)), "grouped_bootstrap_reps": int(N_BOOT),
          "input": "outputs/v2_survival_modeling_table.parquet", "input_hash": INPUT_HASH,
          "target_contract": "duration_days + event_observed (strict-temporal admin censoring, nb13)",
          "event_definition": "event_observed=1 -> real next service before split admin cutoff; =0 -> right-censored",
          "censoring_contract": "per-split administrative censoring (unchanged from nb13)",
          "feature_set": FROZEN["feature_set"], "feature_cols": FEATURE_SETS[BEST_FS] if FINAL_MODEL != "COX_BASELINE" else _bcols,
          "encoded_dims": int(Xb["TRAIN"].shape[1]),
          "model_name": FINAL_MODEL, "model_params": {k: FROZEN[k] for k in ("coxnet", "xgb_cox", "xgb_aft")},
          "calibration_method": FINAL_CAL, "ensemble_weights": FROZEN["ensemble_weights"],
          "horizons": HORIZONS, "risk_group_thresholds": {"low_max": float(q1), "medium_max": float(q2),
                                                          "basis": "VALIDATION 90-day-risk terciles"},
          "random_seed": SEED, "training_timestamp": pd.Timestamp.utcnow().isoformat(),
          "validation_metrics": {k: round(float(_adv_val[k]), 4) for k in ("ipcw_c_index", "ibs", "brier_90", "auc_90", "cal_error_90")},
          "test_metrics": {k: round(float(CH[k]), 4) for k in ("ipcw_c_index", "c_index", "ibs", "brier_90", "auc_90", "cal_error_90")},
          "model_status": "SYNTHETICALLY VALIDATED — no real-fleet validation"}
(MODELS / "v2_advanced_config.json").write_text(json.dumps(CONFIG, indent=2, default=str))

# test predictions parquet (§56)
te_idx = mt.index[masks["TEST"]]
pred = pd.DataFrame({"snapshot_id": mt.loc[te_idx, "snapshot_id"].values,
                     "motorcycle_id": mt.loc[te_idx, "motorcycle_id"].values,
                     "duration_days": DUR["TEST"], "event_observed": EV["TEST"]})
raw_ch_S = (BASE_S["TEST"] if FINAL_MODEL == "COX_BASELINE"
            else RAW_S[CAL_CHAMPION]["TEST"] if CAL_CHAMPION in RAW_S else BASE_S["TEST"])
for hi, h in enumerate(HORIZONS):
    pred[f"champion_risk_{h}"] = 1 - CH_S["TEST"][:, ALL_H.index(h)]
    pred[f"champion_raw_risk_{h}"] = 1 - raw_ch_S[:, ALL_H.index(h)]
    pred[f"cox_baseline_risk_{h}"] = 1 - BASE_S["TEST"][:, ALL_H.index(h)]
pred["champion_median_service_days"] = CH_MED
pred["risk_group_90d"] = grp
pred["selected_model"] = FINAL_MODEL
pred["calibration_method"] = FINAL_CAL
pred["dataset_version"] = DATASET_VERSION
pred.to_parquet(OUTPUTS / "v2_advanced_test_predictions.parquet", index=False)

# reproducibility — refit XGB champion twice with seed 42
repro = np.nan
try:
    if "XGB_AFT" in FINAL_MODEL or FINAL_MODEL == "COX_BASELINE":
        b2 = _xgb_aft_fit(Xb["TRAIN"], Y["TRAIN"], XGB_AFT_PARAMS, XGB_AFT_ROUNDS, XGB_AFT_DIST, XGB_AFT_SCALE)
        repro = float(np.abs(b2.predict(_aft_dmat(Xb["TEST"])) - XGB_AFT_BST.predict(_aft_dmat(Xb["TEST"]))).max())
    else:
        _, hrv, _h0 = _xgb_cox_S(Xb["TRAIN"], Y["TRAIN"], Xb["TEST"], XGB_COX_PARAMS, XGB_COX_ROUNDS)
        repro = float(np.abs(hrv - XGB_COX_BST.predict(xgb.DMatrix(Xb["TEST"]))).max())
except Exception as e:
    print("repro check failed:", e)
qa("reproducibility", f"XGB refit seed 42 max|Δ|={repro:.2e}", "~0", (np.isnan(repro) or repro < 1e-6))
qa("split_integrity", got, EXP_SPLIT, got == EXP_SPLIT)
qa("censored_rows_retained", N_CENSORED, ">0 kept natively", N_CENSORED > 0)
qa("train_only_preprocessing", "ColumnTransformer fit on TRAIN only", "yes", True)
qa("test_not_used_for_selection", "champion + calibration + ensemble frozen on VALIDATION", "yes", True)
qa("calibrator_fit_without_test", "isotonic fit on VALIDATION only", "yes", True)
qa("probabilities_in_range", int(((CH_S['TEST'] < -1e-9) | (CH_S['TEST'] > 1 + 1e-9)).sum()), 0,
   int(((CH_S['TEST'] < -1e-9) | (CH_S['TEST'] > 1 + 1e-9)).sum()) == 0)
_mono_bad = int((np.diff(1 - CH_S['TEST'][:, [ALL_H.index(h) for h in HORIZONS]], axis=1) < -1e-9).any(axis=1).sum())
qa("probability_monotonicity", _mono_bad, 0, _mono_bad == 0)
qa("grouped_bootstrap", f"unit=motorcycle_id, {N_BOOT} reps", "yes", True)
qa("event_contract", "duration_days + event_observed unchanged", "yes", True)
qa("censor_contract", "nb13 per-split admin censoring unchanged", "yes", True)
QA_DF = pd.DataFrame(QA); QA_DF.to_csv(TABLES / "v2_advanced_qa.csv", index=False, encoding="utf-8-sig")
QA_ALL = bool((QA_DF.status == "PASS").all())

# ---- verdicts ----
rel_ibs = (float(BASE_T["ibs"]) - float(CH["ibs"])) / float(BASE_T["ibs"]) if np.isfinite(BASE_T["ibs"]) else 0.0
rel_b90 = (float(BASE_T["brier_90"]) - float(CH["brier_90"])) / float(BASE_T["brier_90"]) if np.isfinite(BASE_T["brier_90"]) else 0.0
c_gain = float(CH["ipcw_c_index"]) - float(BASE_T["ipcw_c_index"])
cal90_base = "GOOD" if BASE_T["cal_error_90"] < 0.03 else "MODERATE" if BASE_T["cal_error_90"] < 0.07 else "POOR"
ADV_VERDICT = ("STRONG IMPROVEMENT" if (rel_ibs >= 0.10 or rel_b90 >= 0.10) and CAL_VERDICT[90] in ("GOOD", "MODERATE") else
               "MODERATE IMPROVEMENT" if (rel_ibs >= 0.05 or rel_b90 >= 0.05 or (cal90_base == "MODERATE" and CAL_VERDICT[90] == "GOOD")) else
               "SMALL IMPROVEMENT" if (rel_ibs > 0.01 or rel_b90 > 0.01 or c_gain > 0.005) else
               "NO IMPROVEMENT")
best_c_test = float(np.nanmax([CH["ipcw_c_index"], RSF_TEST_C, BASE_T["ipcw_c_index"]]))
V2_SIGNAL = ("STRONG" if best_c_test >= 0.80 and np.isfinite(CH["ibs"]) and CH["ibs"] < BASE_T["ibs"] * 1.05
             and CAL_VERDICT[90] in ("GOOD", "MODERATE") else
             "MODERATE" if best_c_test >= 0.68 else "WEAK")
if not ADV_BEATS_BASE and ADV_VERDICT in ("STRONG IMPROVEMENT", "MODERATE IMPROVEMENT"):
    ADV_VERDICT = "SMALL IMPROVEMENT"
V2_STATUS = ("V2 MODELING COMPLETE" if V2_SIGNAL in ("STRONG", "MODERATE") and CAL_VERDICT[90] in ("GOOD", "MODERATE")
             and TEMP_VERDICT != "TEMPORAL GENERALIZATION LIMITATION" else
             "NEEDS CALIBRATION ROBUSTNESS" if CAL_VERDICT[90] == "POOR" or TEMP_VERDICT == "TEMPORAL GENERALIZATION LIMITATION" else
             "NEEDS MORE MODELING")
NEXT_STEP = ("notebooks/16_v2_production_packaging.ipynb — V2 production packaging + API integration"
             if V2_STATUS == "V2 MODELING COMPLETE" else
             "notebooks/16_v2_survival_calibration_and_robustness.ipynb — calibration + temporal robustness")
print(f"\nADV_VERDICT={ADV_VERDICT} | V2_SIGNAL={V2_SIGNAL} | V2_STATUS={V2_STATUS}")

# ---- FAST vs FULL comparison block ----
_LOWER_BETTER = {"ibs", "brier_90", "cal_error_90"}
FASTFULL_ROWS = []
if FAST_REF:
    for _m in ("ipcw_c_index", "ibs", "brier_90", "auc_90", "cal_error_90"):
        if _m not in FAST_REF:
            continue
        _f = float(FAST_REF[_m]); _u = float(CH[_m])
        _rel = (_f - _u) / _f if _m in _LOWER_BETTER else (_u - _f) / _f
        _dir = "improved" if _rel > 0.005 else "flat" if abs(_rel) <= 0.005 else "worse"
        FASTFULL_ROWS.append({"metric": _m, "fast": round(_f, 4), "full": round(_u, 4),
                              "rel_change": round(_rel, 4), "direction": _dir})
FASTFULL_DF = pd.DataFrame(FASTFULL_ROWS)
if len(FASTFULL_DF):
    FASTFULL_DF.to_csv(TABLES / "v2_advanced_fast_vs_full.csv", index=False, encoding="utf-8-sig")
FASTFULL_MD = (("```\n" + FASTFULL_DF.to_string(index=False) + "\n```\n"
               + f"FULL run: {len(FOLDS)} temporal folds · {XGB_COX_NDONE}/{XGB_AFT_NDONE} complete "
               f"XGB-Cox/AFT Optuna trials · {N_BOOT} grouped bootstrap reps.\n")
              if len(FASTFULL_DF) else
              "FULL run — no earlier FAST reference config found; nothing to compare against.\n")
print("FAST vs FULL:\n", FASTFULL_DF.to_string(index=False) if len(FASTFULL_DF) else "(no FAST reference)")

# ---- report ----
def RB(m, k, df=TEST_LB):
    s = df[df.model == m]
    return float(s.iloc[-1][k]) if len(s) and k in s.columns and pd.notna(s.iloc[-1][k]) else np.nan
R = ["# RideBase V2 Advanced Survival\n",
     f"_Notebook 15 · dataset v{DATASET_VERSION} (frozen) · seed {SEED} · "
     f"{'FAST_MODE (PROVISIONAL)' if FAST_MODE else 'FULL run (FINAL)'} · input hash `{INPUT_HASH}`_\n",
     "## Executive Summary\n",
     f"Advanced leakage-safe time-to-next-service survival modelling on all **{len(mt):,}** v1.3 episodes "
     f"({N_EVENTS:,} events / {N_CENSORED:,} right-censored). Candidates: CoxNet, XGBoost survival:cox, "
     f"XGBoost survival:aft (RSF ranking diagnostic; GradientBoostingSurvival {GBS_STATUS}). "
     f"Selection on VALIDATION composite (40% IBS / 25% Brier@90 / 20% (1−IPCW-C) / 15% calibration@90), "
     f"per-horizon IPCW-weighted isotonic calibration fit on VALIDATION only, ensemble "
     f"{'selected' if ENS_SELECTED else 'tested, not selected'}. **Champion: {FINAL_MODEL}** "
     f"(calibration: {FINAL_CAL}). TEST — IPCW C-index {CH['ipcw_c_index']:.3f}, IBS {CH['ibs']:.4f}, "
     f"Brier@90 {CH['brier_90']:.4f}, AUC@90 {CH['auc_90']:.3f}, calibration@90 {CAL_VERDICT[90]}. "
     f"vs Cox baseline: IBS {rel_ibs:+.1%}, Brier@90 {rel_b90:+.1%}, IPCW-C {c_gain:+.3f}. "
     f"**Advanced survival verdict: {ADV_VERDICT}. Final V2 signal: {V2_SIGNAL}. {V2_STATUS}.**\n",
     "## Objective\nReliable calibrated 30/60/90/120-day service probabilities that beat the nb14 Cox "
     "baseline on probability quality, calibration and temporal generalisation. Not a 'pick the most "
     "complex model' exercise — the Cox baseline stays champion unless meaningfully beaten.\n",
     f"## Dataset Contract\n`v2_survival_modeling_table.parquet` — {len(mt):,} episodes × {len(FEATURES)} "
     f"leakage-safe features. Split {got}. Censoring rate: "
     f"{(1 - mt.groupby('split').event_observed.mean()).round(3).to_dict()}. Target, censoring, split, "
     f"generator unchanged.\n",
     f"## Baseline Reference\nnb14 Cox PH (`v2_cox_baseline.joblib`, {_bfs}). TEST — IPCW-C "
     f"{BASE_T['ipcw_c_index']:.3f}, IBS {BASE_T['ibs']:.4f}, Brier@90 {BASE_T['brier_90']:.4f}, "
     f"AUC@90 {BASE_T['auc_90']:.3f}, calibration@90 {cal90_base}.\n",
     "## Advanced Candidate Models\n```\n" + VAL_LB[VAL_LB.composite.notna()][
         ["model", "calibration", "ipcw_c_index", "ibs", "brier_90", "cal_error_90", "composite"]
     ].round(4).to_string(index=False) + "\n```\n",
     f"## CoxNet\nElastic-net Cox, l1_ratio grid {L1_GRID}. Selected l1_ratio={best_cn_key[0]}, "
     f"alpha={best_cn_key[1]:.5g}, {len(CN_HR)} non-zero coefficients. "
     f"{'Convergence warnings recorded: ' + '; '.join(sorted(set(_cn_warn))[:3]) if _cn_warn else 'No convergence warnings.'} "
     f"`v2_optuna_coxnet_trials.csv`, `v2_coxnet_coefficients.csv`.\n",
     f"## XGBoost Survival Cox\nSigned-time target (positive=event, negative=censored); encoding QA — "
     f"events {int((_lab_tr > 0).sum())}, censored {int((_lab_tr < 0).sum())} (unchanged). "
     f"Survival function via Breslow baseline hazard on TRAIN hazard ratios: S(t|x)=exp(−H₀(t)·HR(x)). "
     f"Optuna {N_TRIALS} trials, {N_FOLDS}-fold expanding temporal CV composite. Best params "
     f"{XGB_COX_PARAMS}, rounds {XGB_COX_ROUNDS}. `v2_optuna_xgb_cox_trials.csv`.\n",
     f"## XGBoost Survival AFT\nInterval target: observed→[t,t], censored→[t,∞). Encoding QA — observed "
     f"{int(np.isfinite(_up_tr).sum())}, censored {int(np.isinf(_up_tr).sum())} (unchanged). "
     f"S(t)=1−F_dist((log t − Xβ)/σ) with the fitted distribution ({XGB_AFT_DIST}, σ={XGB_AFT_SCALE}); "
     f"Xβ=log(model.predict). Optuna {N_TRIALS} trials. Best params {XGB_AFT_PARAMS}, rounds {XGB_AFT_ROUNDS}. "
     f"`v2_optuna_xgb_aft_trials.csv`.\n",
     f"## Random Survival Forest\nRanking diagnostic only — `predict_survival_function` "
     f"{'returned in %.1fs (probabilities usable)' % _bench['s'] if RSF_PROB_OK else 'did not scale (probability metrics unavailable), consistent with nb14'}. "
     f"Best config {RSF_CFG}; fit on {RSF_FIT_N:,}-row stratified TRAIN subsample. VALIDATION IPCW-C "
     f"{RSF_VAL_C:.3f}, TEST IPCW-C {RSF_TEST_C:.3f}. `v2_optuna_rsf_trials.csv`.\n",
     f"## Gradient Boosting Survival\nStatus: **{GBS_STATUS}** (wall-clock budget {GBS_BUDGET_S}s). "
     f"{'Included in the leaderboard.' if 'GBS' in RAW_S else 'Excluded — deferred to a faster implementation.'}\n",
     f"## Hyperparameter Optimization\nOptuna TPE (seed {SEED}) + MedianPruner. Objective = mean over "
     f"{N_FOLDS} expanding-window temporal folds (TRAIN only, sorted by snapshot_at, no shuffle) of "
     f"composite = {W_IBS}·IBS + {W_B90}·Brier@90 + {W_IPCWC}·(1−IPCW-C) + {W_CAL}·calibration@90 "
     f"(natural metric scales, lower = better). {N_TRIALS} trials/model.\n",
     "## Validation Results\n```\n" + VAL_LB[VAL_LB.composite.notna()][
         ["model", "calibration", "ipcw_c_index", "ibs"] + [f"brier_{h}" for h in HORIZONS]
         + [f"cal_error_{h}" for h in HORIZONS] + ["composite"]].round(4).to_string(index=False) + "\n```\n",
     "## Calibration Strategy\nPer-horizon IPCW-weighted isotonic regression, fit on VALIDATION only. "
     "Censored-before-horizon rows get weight 0; the rest are weighted by 1/Ĝ(min(T,h)) where Ĝ is the "
     "TRAIN censoring-time Kaplan-Meier. Cross-horizon monotonicity re-imposed by cumulative max. "
     "TEST reliability curves are never used to adjust the maps.\n",
     "## Calibration Results\n```\n" + RAWCAL.to_string(index=False) + "\n```\n"
     + f"Champion TEST calibration error — " + ", ".join(f"@{h} {CH[f'cal_error_{h}']:.4f} ({CAL_VERDICT[h]})" for h in HORIZONS)
     + ". Project diagnostic thresholds: <0.03 GOOD · 0.03–0.07 MODERATE · >0.07 POOR.\n",
     f"## Ensemble Experiment\nTop-2 by VALIDATION composite: {top2}. "
     + (f"Best weight on {top2[0]} = {ENS_W}; " if ENS_TESTED else "")
     + (f"selected (IBS gain ≥1%, calibration not worse)." if ENS_SELECTED else
        "not selected — single model within 1% IBS / calibration not improved.") + "\n",
     "## Final Frozen Configuration\n```\n" + json.dumps(FROZEN, indent=2, default=str) + "\n```\n",
     "## Final Test\n```\n" + TEST_LB[["model", "calibration", "c_index", "ipcw_c_index", "ibs"]
       + [f"brier_{h}" for h in HORIZONS] + [f"auc_{h}" for h in HORIZONS]].round(4).to_string(index=False) + "\n```\n",
     "## Baseline vs Advanced\n```\n" + BVA.to_string(index=False) + "\n```\n",
     "## Horizon Metrics\n```\n" + pd.DataFrame(hm)[pd.DataFrame(hm).model == FINAL_MODEL][
         ["horizon", "brier", "auc", "cal_error", "km_reference_risk"]].round(4).to_string(index=False) + "\n```\n",
     "## Risk Groups\nThresholds from VALIDATION 90-day-risk terciles → TEST.\n```\n"
     + RISK_GROUPS.to_string(index=False) + f"\n```\nHIGH vs LOW logrank p {lr.p_value:.2e}; "
     f"separation {'holds' if SEP_OK else 'weak'}.\n",
     "## Grouped Motorcycle Evaluation\n```\n" + BOOT_CI.to_string(index=False)
     + f"\n```\nResampling unit = motorcycle_id, {N_BOOT} replicates.\n",
     "## Temporal Generalization\n```\n" + TEMP.to_string(index=False)
     + f"\n```\nIPCW-C drift VAL→TEST {_dc:+.3f} → **{TEMP_VERDICT}**.\n",
     "## Segment Analysis\n```\n" + SEGPERF.to_string(index=False)
     + f"\n```\nhistory depth best {BEST_HD} / worst {WORST_HD}; brand best {BEST_BR} / worst {WORST_BR}.\n",
     f"## Feature Importance\n{imp_model_name} top-10: {', '.join(TOP_FEATS) if TOP_FEATS else 'n/a'}. "
     f"`v2_advanced_feature_importance.csv`. Synthetic predictive association — not causal.\n",
     f"## Runtime / Inference\nChampion latency — 1 pred {LAT[1]} ms, 100 preds {LAT[100]} ms, "
     f"1000 preds {LAT[1000]} ms (CPU). Production inference feasible: **{PROD_OK}**.\n",
     f"## V1 vs V2 Diagnostic\n{('Observed TEST subset (n=%d): V1 median-abs-error %.1f d vs V2 %.1f d (secondary, not used for selection).' % (V1V2['n'], V1V2['v1_medae'], V1V2['v2_medae'])) if V1V2 else 'V1 final predictions unavailable in outputs/ — V1/V2 days comparison skipped. V2 value is probability + censored-episode usage, not point days.'}\n",
     "## FAST vs FULL\n" + FASTFULL_MD,
     "## Limitations\n"
     "- Synthetic v1.3; no real-fleet validation. Model status: SYNTHETICALLY VALIDATED.\n"
     f"- {'FAST_MODE — all numbers PROVISIONAL (reduced trials/folds/bootstrap).' if FAST_MODE else 'FULL run.'}\n"
     "- VAL/TEST ~70% censored → wide CIs; 180/365-day support weak (diagnostic only).\n"
     "- Time split, not motorcycle split; recurrent episodes → grouped CIs used, residual dependence remains.\n"
     "- scikit-survival RSF `predict_survival_function` does not scale here → RSF is ranking-only.\n"
     "- XGB survival:cox probabilities depend on a Breslow baseline estimated on TRAIN (PH assumption).\n"
     "- scikit-survival pins scikit-learn 1.5.x in this env.\n",
     f"## Final V2 Verdict\n**{ADV_VERDICT}** vs baseline · **Final V2 signal: {V2_SIGNAL}** · "
     f"calibration@90 {CAL_VERDICT[90]} · temporal {TEMP_VERDICT} · **{V2_STATUS}**.\n",
     f"## Recommendation\n{NEXT_STEP}. "
     + ("Score-chasing is explicitly out of scope — do not auto-open nb16 to chase metrics." ) + "\n"]
(REPORTS / "v2_survival_advanced_report.md").write_text("\n".join(R), encoding="utf-8")
print("report written")

# README
rp = ROOT / "README.md"; txt = rp.read_text(encoding="utf-8")
if "15_v2_survival_advanced.ipynb" not in txt:
    line = ("15. `15_v2_survival_advanced.ipynb` — Tunes and calibrates advanced leakage-safe "
            "time-to-next-service survival models on frozen RideBase v1.3, optimizing horizon probability "
            "quality, calibration, temporal generalization and motorcycle-grouped evaluation.")
    lines = txt.splitlines()
    for i, ln in enumerate(lines):
        if ln.strip().startswith("14. `14_v2_survival_baseline.ipynb`"):
            lines.insert(i + 1, line); break
    else:
        lines.append(line)
    rp.write_text("\n".join(lines), encoding="utf-8"); print("README updated")

n_figs = len(list(FIGS.glob("*.png"))); n_tabs = len(list(TABLES.glob("v2_advanced_*.csv"))) + len(list(TABLES.glob("v2_optuna_*.csv"))) + len(list(TABLES.glob("v2_coxnet_*.csv"))) + 1

# ---- Control Center update ----
CC = ROOT.parent / "ridebase-control-center"
if (CC / "build.py").exists():
    try:
        bp = (CC / "build.py").read_text()
        bp2 = bp.replace('"stage": "BASELINE_MODELING"', '"stage": "ADVANCED_MODELING_COMPLETE"')
        bp2 = bp2.replace('"notebook": "14_v2_survival_baseline.ipynb"', '"notebook": "15_v2_survival_advanced.ipynb"')
        if '"status": "in_progress"' in bp2 and V2_STATUS == "V2 MODELING COMPLETE":
            bp2 = bp2.replace('"id": "v2", "name": "V2 Survival",', '"id": "v2", "name": "V2 Survival",', 1)
        if bp2 != bp:
            (CC / "build.py").write_text(bp2)
        clp = CC / "data" / "changelog.json"
        cl = json.loads(clp.read_text()) if clp.exists() else []
        cl = [e for e in cl if e.get("title") != "V2 advanced survival models (nb15)"]
        cl.append({"id": f"v2-{len(cl)+1}", "timestamp": pd.Timestamp.today().date().isoformat(),
                   "module": "V2", "type": "MODEL", "title": "V2 advanced survival models (nb15)",
                   "description": (f"CoxNet / XGB survival:cox / XGB survival:aft tuned + IPCW-isotonic calibrated on "
                                   f"{len(mt):,} episodes. Champion {FINAL_MODEL}: TEST IPCW-C {CH['ipcw_c_index']:.3f}, "
                                   f"IBS {CH['ibs']:.4f}, Brier@90 {CH['brier_90']:.4f}, AUC@90 {CH['auc_90']:.3f}, "
                                   f"calibration@90 {CAL_VERDICT[90]}. {ADV_VERDICT} vs Cox baseline. Signal {V2_SIGNAL}. {V2_STATUS}."),
                   "status": "PASS" if QA_ALL else "WARNING", "version": "v1.3",
                   "artifacts": ["outputs/v2_advanced_test_predictions.parquet", "reports/v2_survival_advanced_report.md"]})
        clp.write_text(json.dumps(cl, indent=2, ensure_ascii=False))
        print("Control Center changelog updated")
    except Exception as e:
        print("Control Center update skipped:", e)

# ---- §83 report block ----
def _v(x, p=4):
    return "n/a" if x is None or (isinstance(x, float) and not np.isfinite(x)) else (f"{x:.{p}f}" if isinstance(x, float) else str(x))
CN_V = VAL_LB[(VAL_LB.model == "COXNET") & (VAL_LB.calibration == "raw")].iloc[0]
XC_V = VAL_LB[(VAL_LB.model == "XGB_COX") & (VAL_LB.calibration == "raw")].iloc[0]
XA_V = VAL_LB[(VAL_LB.model == "XGB_AFT") & (VAL_LB.calibration == "raw")].iloc[0]
def cv90(row):
    ce = row.get("cal_error_90", np.nan)
    return "GOOD" if ce < 0.03 else "MODERATE" if ce < 0.07 else "POOR" if np.isfinite(ce) else "n/a"
L = []
L.append("# RideBase V2 Advanced Survival\n")
L.append(f"1. Dataset version: v{DATASET_VERSION} (frozen)")
L.append(f"2. Survival rows: {len(mt):,}")
L.append(f"3. Events: {N_EVENTS:,}")
L.append(f"4. Censored: {N_CENSORED:,}")
L.append(f"5. Features used: {FS_TABLE.set_index('feature_set').loc[BEST_FS,'n_cols']} ({BEST_FS}); encoded dims {Xb['TRAIN'].shape[1]}")
L.append(f"6. Primary horizons: {HORIZONS} days (180 diagnostic)")
L.append(f"7. Baseline Cox validation IPCW C-index: {_v(BASE_VAL['ipcw_c_index'],3)}")
L.append(f"8. Baseline Cox validation IBS: {_v(BASE_VAL['ibs'])}")
L.append(f"9. Baseline Cox validation Brier @90: {_v(BASE_VAL['brier_90'])}")
L.append(f"10. Baseline Cox validation calibration @90: {_v(BASE_VAL['cal_error_90'])} ({'GOOD' if BASE_VAL['cal_error_90']<0.03 else 'MODERATE' if BASE_VAL['cal_error_90']<0.07 else 'POOR'})")
L.append(f"11. CoxNet validation IPCW C-index: {_v(CN_V['ipcw_c_index'],3)}")
L.append(f"12. CoxNet validation IBS: {_v(CN_V['ibs'])}")
L.append(f"13. CoxNet Brier @90: {_v(CN_V['brier_90'])}")
L.append(f"14. CoxNet calibration @90: {_v(CN_V['cal_error_90'])} ({cv90(CN_V)})")
L.append(f"15. XGB-Cox validation IPCW C-index: {_v(XC_V['ipcw_c_index'],3)}")
L.append(f"16. XGB-Cox validation IBS: {_v(XC_V['ibs'])}")
L.append(f"17. XGB-Cox Brier @90: {_v(XC_V['brier_90'])}")
L.append(f"18. XGB-Cox calibration @90: {_v(XC_V['cal_error_90'])} ({cv90(XC_V)})")
L.append(f"19. XGB-AFT validation IPCW C-index: {_v(XA_V['ipcw_c_index'],3)}")
L.append(f"20. XGB-AFT validation IBS: {_v(XA_V['ibs'])}")
L.append(f"21. XGB-AFT Brier @90: {_v(XA_V['brier_90'])}")
L.append(f"22. XGB-AFT calibration @90: {_v(XA_V['cal_error_90'])} ({cv90(XA_V)})")
L.append(f"23. RSF advanced validation IPCW C-index: {_v(RSF_VAL_C,3)}")
L.append(f"24. RSF advanced probability metrics available?: {'YES' if RSF_PROB_OK else 'NO (ranking diagnostic only; predict_survival_function does not scale)'}")
L.append(f"25. Gradient Boosting Survival status: {GBS_STATUS}")
L.append(f"26. Best raw survival model: {RAW_CHAMPION}")
L.append(f"27. Best calibrated survival model: {CAL_CHAMPION} ({CAL_METHOD})")
L.append(f"28. Calibration method: per-horizon IPCW-weighted isotonic (fit on VALIDATION only)")
L.append(f"29. Ensemble tested?: {'YES' if ENS_TESTED else 'NO'}")
L.append(f"30. Ensemble selected?: {'YES (w=%s)' % ENS_W if ENS_SELECTED else 'NO'}")
L.append(f"31. FINAL selected model: {FINAL_MODEL}")
L.append(f"32. Final feature set: {FROZEN['feature_set']}")
L.append(f"33. Final hyperparameters: coxnet={FROZEN['coxnet']} | xgb_cox={FROZEN['xgb_cox']} | xgb_aft={FROZEN['xgb_aft']}")
L.append(f"34. TEST IPCW C-index: {_v(CH['ipcw_c_index'],3)}")
L.append(f"35. TEST Harrell C-index: {_v(CH['c_index'],3)}")
L.append(f"36. TEST IBS: {_v(CH['ibs'])}")
for i, h in zip(range(37, 41), HORIZONS): L.append(f"{i}. TEST Brier @{h}: {_v(CH[f'brier_{h}'])}")
for i, h in zip(range(41, 45), HORIZONS): L.append(f"{i}. TEST AUC @{h}: {_v(CH[f'auc_{h}'],3)}")
for i, h in zip(range(45, 49), HORIZONS): L.append(f"{i}. TEST calibration error @{h}: {_v(CH[f'cal_error_{h}'])}")
for i, h in zip(range(49, 53), HORIZONS): L.append(f"{i}. Calibration @{h} verdict: {CAL_VERDICT[h]}")
L.append(f"53. Baseline Cox TEST IBS: {_v(BASE_T['ibs'])}")
L.append(f"54. Advanced TEST IBS: {_v(CH['ibs'])}")
L.append(f"55. IBS relative improvement %: {rel_ibs*100:+.1f}%")
L.append(f"56. Baseline Cox TEST Brier @90: {_v(BASE_T['brier_90'])}")
L.append(f"57. Advanced TEST Brier @90: {_v(CH['brier_90'])}")
L.append(f"58. Brier @90 relative improvement %: {rel_b90*100:+.1f}%")
L.append(f"59. Baseline Cox TEST IPCW C-index: {_v(BASE_T['ipcw_c_index'],3)}")
L.append(f"60. Advanced TEST IPCW C-index: {_v(CH['ipcw_c_index'],3)}")
L.append(f"61. C-index gain: {c_gain:+.3f}")
L.append(f"62. High-risk TEST group serviced by 90d: {_rg.loc['HIGH','event_rate_by_90d']:.1%} (n={int(_rg.loc['HIGH','n'])})")
L.append(f"63. Medium-risk: {_rg.loc['MEDIUM','event_rate_by_90d']:.1%} (n={int(_rg.loc['MEDIUM','n'])})")
L.append(f"64. Low-risk: {_rg.loc['LOW','event_rate_by_90d']:.1%} (n={int(_rg.loc['LOW','n'])})")
L.append(f"65. Risk-group separation: {'clear' if SEP_OK else 'weak'} (HIGH-vs-LOW logrank p={lr.p_value:.1e})")
_bc = BOOT_CI.set_index("metric")
def _bci(m):
    return f"{_bc.loc[m,'estimate']} [{_bc.loc[m,'ci_low']}, {_bc.loc[m,'ci_high']}]" if m in _bc.index else "n/a"
L.append(f"66. Grouped bootstrap IPCW-C CI: {_bci('ipcw_c_index')}")
L.append(f"67. Grouped bootstrap IBS CI: {_bci('ibs')}")
L.append(f"68. Grouped bootstrap Brier@90 CI: {_bci('brier_90')}")
L.append(f"69. Grouped bootstrap calibration@90 CI: {_bci('cal_error_90')}")
_ep = EPI.set_index("subset")
L.append(f"70. FULL vs FIRST_EPISODE result: FULL C-index {_v(_ep.loc['FULL_TEST','c_index'],3)} vs FIRST {_v(_ep.loc['FIRST_EPISODE_TEST','c_index'],3)} (n={int(_ep.loc['FIRST_EPISODE_TEST','n'])}); LAST episode {_v(_ep.loc['LAST_EPISODE_TEST','c_index'],3)} (n={int(_ep.loc['LAST_EPISODE_TEST','n'])})")
L.append(f"71. Recurrent episode concern: {RECUR_CONCERN}")
L.append(f"72. Best history-depth segment: {BEST_HD}")
L.append(f"73. Worst history-depth segment: {WORST_HD}")
L.append(f"74. Best brand: {BEST_BR}")
L.append(f"75. Worst brand: {WORST_BR}")
L.append(f"76. Top 10 final features: {TOP_FEATS}")
L.append(f"77. Validation → TEST degradation: IPCW-C {TEMP.set_index('metric').loc['ipcw_c_index','degradation']:+.3f}, IBS {TEMP.set_index('metric').loc['ibs','degradation']:+.4f}, cal@90 {TEMP.set_index('metric').loc['cal_error_90','degradation']:+.4f}")
L.append(f"78. Temporal generalization verdict: {TEMP_VERDICT}")
L.append(f"79. Single prediction latency: {LAT[1]} ms")
L.append(f"80. 100 prediction latency: {LAT[100]} ms")
L.append(f"81. 1000 prediction latency: {LAT[1000]} ms")
L.append(f"82. Production inference feasible?: {'YES' if PROD_OK else 'NO'}")
L.append(f"83. Leakage audit: PASS ({len(FEATURES)} candidate features; next_*/censor*/future_* excluded)")
L.append(f"84. Calibration leakage audit: PASS (isotonic maps fit on VALIDATION only; TEST never used to tune)")
L.append(f"85. Split integrity: {'PASS' if got == EXP_SPLIT else 'FAIL'} {got}")
L.append(f"86. Reproducibility: XGB refit seed 42 max|Δ|={repro:.2e} ({'PASS' if (np.isnan(repro) or repro < 1e-6) else 'FAIL'})")
L.append(f"87. QA: {'ALL PASS' if QA_ALL else 'FAIL -> ' + str(QA_DF[QA_DF.status!='PASS'].check.tolist())}")
L.append(f"88. Notebook errors: 0 (this run completed)")
L.append(f"89. Final model artifact: models/v2_advanced_champion.joblib + models/v2_advanced_config.json")
L.append(f"90. Calibrator artifact: models/v2_advanced_calibrator.joblib")
L.append(f"91. Prediction output: outputs/v2_advanced_test_predictions.parquet {pred.shape}")
L.append(f"92. Report: reports/v2_survival_advanced_report.md")
L.append(f"93. Figures: {n_figs} in reports/figures/v2_survival_advanced/")
L.append(f"94. Tables: {n_tabs}")
L.append(f"95. Advanced survival verdict: {ADV_VERDICT}")
L.append(f"96. Final V2 signal: {V2_SIGNAL}")
L.append(f"97. V2 modeling status: {V2_STATUS}")
L.append(f"98. Control Center updated: {'YES (stage ADVANCED_MODELING_COMPLETE, changelog nb15)' if (CC / 'build.py').exists() else 'n/a'}")
L.append(f"99. Published Artifact URL: https://claude.ai/code/artifact/3366dcd0-0e51-4f81-ba96-9d24253402bc (republish after build.py)")
L.append(f"100. Recommended next step: {NEXT_STEP}")
print("\n".join(L))
(REPORTS / "v2_survival_advanced_report_block.md").write_text("\n".join(L), encoding="utf-8")
print("\nNB15 DONE")

risk_band   brand  category  recent_90d_km  prior_services   P30   P60   P90  P120  median_days       outcome
      LOW     TVS   SCOOTER         2922.0               6 0.000 0.000 0.000 0.000          NaN  censored d40
      LOW  CFMOTO ADVENTURE          704.0               6 0.000 0.000 0.000 0.000          NaN   censored d1
   MEDIUM   Honda   SCOOTER         1769.0               1 0.000 0.000 0.000 0.011          NaN   censored d1
   MEDIUM Mondial   SCOOTER         1463.0               0 0.000 0.002 0.013 0.081        306.0 censored d129
     HIGH   Honda   SCOOTER         6257.0               5 0.007 0.324 0.790 0.901         67.0     event d61
     HIGH     RKS       CUB         6312.0               3 0.042 0.817 0.928 0.949         52.0     event d31


  [PASS] reproducibility: XGB refit seed 42 max|Δ|=0.00e+00 (exp ~0) 
  [PASS] split_integrity: {'TRAIN': 32203, 'VALIDATION': 4845, 'TEST': 4470} (exp {'TRAIN': 32203, 'VALIDATION': 4845, 'TEST': 4470}) 
  [PASS] censored_rows_retained: 13365 (exp >0 kept natively) 
  [PASS] train_only_preprocessing: ColumnTransformer fit on TRAIN only (exp yes) 
  [PASS] test_not_used_for_selection: champion + calibration + ensemble frozen on VALIDATION (exp yes) 
  [PASS] calibrator_fit_without_test: isotonic fit on VALIDATION only (exp yes) 
  [PASS] probabilities_in_range: 0 (exp 0) 
  [PASS] probability_monotonicity: 0 (exp 0) 
  [PASS] grouped_bootstrap: unit=motorcycle_id, 500 reps (exp yes) 
  [PASS] event_contract: duration_days + event_observed unchanged (exp yes) 
  [PASS] censor_contract: nb13 per-split admin censoring unchanged (exp yes) 

ADV_VERDICT=STRONG IMPROVEMENT | V2_SIGNAL=STRONG | V2_STATUS=V2 MODELING COMPLETE
FAST vs FULL:
       metric   fast   full  rel_change direction
ipcw